# W12｜語音合成、語音辨識與深偽風險

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/course-115-v2/colab/W12_%E8%AA%9E%E9%9F%B3%E5%90%88%E6%88%90%E8%BE%A8%E8%AD%98%E8%88%87%E6%B7%B1%E5%81%BD.ipynb)

- **對應講義**：[HackMD 第 12 章　語音合成、語音辨識與深偽風險](https://hackmd.io/@wiimax/HJ7jcqY8Gg)
- **互動實驗室**：[W12 Lab（瀏覽器版）](https://willismax.github.io/MediaSystem-Python-Course/labs/w12/)
- **先備**：W10 取樣、STFT／頻譜圖；W11 pydub 剪輯、`detect_nonsilent` 靜音偵測

W10 我們「看懂聲音」，W11 「動手改聲音」，本週要「**生成聲音**」與「**聽懂聲音**」：用 **edge-tts** 與 **ZONOS3-Mini** 做文字轉語音（TTS），用 **Whisper** 做語音辨識（STT），用自己寫的 Levenshtein 對齊計算 **CER／WER**，再把兩者串成 **TTS → STT 往返測試**；最後以「**證據卡**」與「**Audio Disclosure**」面對深偽（deepfake）風險。

| 學習目標 | 你要能做到 |
|---|---|
| TTS 管線 | 說明「文字 → 聲學特徵（Mel）→ vocoder → 波形」，以及為什麼需要 vocoder |
| 控制變因 | 設計只改**一個**變因的盲測，並在揭曉答案**之前**完成評分 |
| Whisper | 在 Colab 本地執行 Whisper、比較模型大小與執行時間，知道時間戳與 `initial_prompt` 的極限 |
| CER／WER | 自己實作 Levenshtein 對齊，並**明確宣告**文字正規化規則 |
| 語者條件 | 分辨 TTS、語者條件 TTS（voice cloning）與語音轉換，理解「聽起來像」不等於「就是本人」 |
| 深偽與揭露 | 用證據卡寫出**信心程度**（而非只寫真／假），並為作品附上 Audio Disclosure |

**使用方式**

- 從上到下依序執行即可。**建議先切換 GPU**：執行階段 → 變更執行階段類型 → **T4 GPU**。Whisper `small` 與 ZONOS3-Mini 在 CPU 上也能跑，但會慢很多。
- 需要網路的步驟：edge-tts（連到 Microsoft Edge 線上朗讀服務）、Whisper 權重下載、ZONOS3-Mini 權重下載（Hugging Face）。任何一步失敗，程式都會**印出原因並改用替代方案繼續**，「全部執行」不會中斷；但替代方案（例如佔位聲音）只是讓流程跑得完，**不能拿來寫報告結論**。筆記本最後的「執行摘要」會列出哪些步驟真的執行、哪些用了替代方案。
- 用到麥克風的格子預設關閉（`RECORD = False`）；聲音複製（voice cloning）**只限你自己的聲音**，而且預設關閉。

> 圖表的座標軸標籤一律使用英文（避免 Colab 缺中文字型而出現方框），說明文字則用中文。

## 本週流程地圖

| 段落 | 主題 | 主要程式 |
|---|---|---|
| 0 | 環境、工具函式、（選用）錄自己的聲音 | `play()`、`load_audio()`、`tts_to_wav()`、`record_colab()` |
| 12.22 | 本週安全邊界（**先讀**）| `SAFETY_BOUNDARY` |
| 12.1、12.3 | TTS 管線、語者條件、語音轉換、語者嵌入 | |
| 12.4 | edge-tts：第一段合成語音、換聲音、調語速 | `edge_tts.Communicate()`、`await speak()` |
| 12.2、12.1 | Mel 頻譜 vs 線性 STFT、Mel 濾波器組、Griffin-Lim 與 vocoder | `librosa.feature.melspectrogram()`、`mel_to_audio()` |
| 12.5 | 控制變因盲測 | `make_blind_test()`、`reveal()` |
| 12.6–12.10 | Whisper：轉錄、時間戳、SRT、`initial_prompt`、簡轉繁 | `whisper.load_model()`、`model.transcribe()` |
| 12.11–12.13 | WER／CER、Levenshtein 對齊、正規化規則 | `align()`、`cer()`、`wer()`、`RULESETS` |
| 12.14 | 控制實驗：兩個 Whisper 模型 | `compare_models()` |
| 12.15 | TTS → STT 往返 | `round_trip()` |
| Z | ZONOS3-Mini：多語音色、seed／temperature、語者嵌入、本人聲音複製、揭露 | `zonos.generate()`、`zonos.embed_speaker()` |
| 12.16–12.20 | 深偽風險、可觀察線索、證據卡 | `audio_clues()`、`evidence_card_md()` |
| 12.23 | Audio Disclosure | `audio_disclosure()` |
| 練習 1–4 | 盲測、Whisper 準確度、往返測試、證據卡 | |
| 延伸 | 自動字幕、CER 與正規化、Speech AI 品質矩陣 | |
| 檢核 | 執行摘要、檢核題、繳交清單 | |

## 0.1 安裝套件

- `edge-tts`：Microsoft Edge 線上「大聲朗讀」服務的**第三方**用戶端，**不是**官方的 Azure Speech SDK。需要網路；聲音名稱可能隨服務更新而變動。
- `openai-whisper`：OpenAI 開源的語音辨識模型，**在 Colab 主機上本地執行**；需要 FFmpeg（Colab 已內建）。
- `jiwer`：只用來**交叉驗證**我們自己寫的 WER／CER。
- `opencc-python-reimplemented`：簡體轉繁體（選用）。
- ZONOS3-Mini 需要的 `transformers` 等套件，到 Z 段再安裝。

> 安裝訊息若出現版本衝突的警告，通常不影響本週內容；若出現錯誤，後面相關的格子會印出提示並改用替代方案。

In [ ]:
!pip -q install edge-tts librosa soundfile jiwer opencc-python-reimplemented
!pip -q install -U openai-whisper

## 0.2 匯入與工具函式

| 函式 | 用途 |
|---|---|
| `load_audio(path, sr=None)` | 讀成單聲道浮點陣列；MP3 等格式先用 ffmpeg 轉成 WAV |
| `play(y 或 路徑, sr)` | 先存成 WAV bytes 再播放，**刻意不用** `Audio(y, rate=sr)`——那種寫法會自動把音量正規化，你就聽不出音量差別了 |
| `tts_to_wav(text, wav, voice, rate)` | edge-tts 合成並轉成 WAV；連不上服務時改用**佔位聲音**（不是 TTS，只是讓後面流程能跑），並記錄在 `GEN_LOG` |
| `show_table(header, rows)` | 以 Markdown 表格顯示結果 |

`STATUS` 記錄每個需要網路或模型的步驟是「真的執行」還是「用了替代方案」；`GEN_LOG` 記錄每段合成語音的引擎、聲音、參數與生成時間，之後盲測、品質矩陣與 Audio Disclosure 都會用到。

In [ ]:
import asyncio
import inspect
import io
import json
import random
import re
import subprocess
import time
import unicodedata
import warnings
from datetime import date
from importlib import metadata
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, Markdown, display

warnings.filterwarnings("ignore", message="invalid value encountered in cast")  # librosa/numba 的無害警告
plt.rcParams.update({"figure.figsize": (11, 3), "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(1202)
TODAY = date.today().isoformat()

STATUS = {}    # 步驟 → "real"（真的執行）／"fallback"（替代方案）／"skipped"
GEN_LOG = {}   # 標籤 → 這段合成語音的引擎、聲音、參數、生成時間、檔案

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "—"
except ImportError:
    torch = None
    DEVICE, GPU_NAME = "cpu", "—（未安裝 torch）"
print(f"運算裝置：{DEVICE}｜GPU：{GPU_NAME}")
if DEVICE == "cpu":
    print("提示：目前是 CPU。Whisper small 與 ZONOS3-Mini 會很慢；"
          "建議：執行階段 → 變更執行階段類型 → T4 GPU，然後重新執行。")


def pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "未安裝"


def to_wav(src, dst, sr=None):
    """用 ffmpeg 把任何音訊轉成單聲道 16-bit WAV（sr=None 表示保留原取樣率）。"""
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-ac", "1"]
    if sr:
        cmd += ["-ar", str(int(sr))]
    cmd += ["-c:a", "pcm_s16le", str(dst)]
    subprocess.run(cmd, check=True)
    return str(dst)


def load_audio(path, sr=None):
    """讀成 (float32 單聲道陣列, 取樣率)。非 WAV／FLAC 先經 ffmpeg 轉檔。"""
    path = Path(path)
    if path.suffix.lower() not in (".wav", ".flac"):
        tmp = path.with_name(path.stem + "_decoded.wav")
        to_wav(path, tmp)
        path = tmp
    y, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    y = y.mean(axis=1)
    if sr and sr != file_sr:
        y = librosa.resample(y, orig_sr=file_sr, target_sr=sr)
        file_sr = sr
    return y, int(file_sr)


def play(y_or_path, sr=None, label=""):
    """以 WAV bytes 原樣播放（不自動正規化音量）。"""
    if label:
        display(Markdown(f"**聆聽：{label}**"))
    if isinstance(y_or_path, (str, Path)):
        y, sr = load_audio(y_or_path)
    else:
        y = np.asarray(y_or_path, dtype=np.float32)
    buf = io.BytesIO()
    sf.write(buf, np.clip(y, -1, 1), int(sr), format="WAV", subtype="PCM_16")
    display(Audio(data=buf.getvalue()))


def show_table(header, rows):
    """以 Markdown 表格顯示結果。"""
    def cell(v):
        s = f"{v:.2f}" if isinstance(v, float) else str(v)
        return s.replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(cell(v) for v in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


def duration(path):
    info = sf.info(str(path))
    return info.frames / info.samplerate


# ---------- 離線佔位聲音（不是 TTS！只讓後續流程在沒有網路時也能執行）----------
_FORMANTS = [(730, 1090), (270, 2290), (300, 870), (530, 1840), (570, 840), (440, 1020)]


def _syllable(dur, f0, sr, r):
    n = int(dur * sr)
    t = np.arange(n) / sr
    phase = 2 * np.pi * np.cumsum(f0 * (1 + 0.05 * np.sin(2 * np.pi * 2 * t))) / sr
    F1, F2 = _FORMANTS[r.integers(len(_FORMANTS))]
    sig = np.zeros(n)
    for k in range(1, 40):
        fk = f0 * k
        if fk > 0.45 * sr:
            break
        w = np.exp(-((fk - F1) / 200) ** 2) + 0.6 * np.exp(-((fk - F2) / 300) ** 2) + 0.03 / k
        sig += w * np.sin(k * phase)
    return sig * np.sin(np.pi * np.linspace(0, 1, n)) ** 0.7


def placeholder_speech(text, sr=24000, seed=0, rate=1.0):
    """每個中文字一個「類母音」音節、英數字約兩字元一個音節、標點變成停頓。"""
    r = np.random.default_rng(seed)
    f0 = r.uniform(110, 230)
    units = re.findall(r"[A-Za-z0-9]{1,2}|[\u4e00-\u9fff]|[，。、；：？！,.;:?!]", text)
    parts = [np.zeros(int(0.2 * sr))]
    for i, u in enumerate(units):
        if re.fullmatch(r"[，。、；：？！,.;:?!]", u):
            parts.append(np.zeros(int(0.25 * sr / rate)))
            continue
        parts.append(_syllable(r.uniform(0.16, 0.24) / rate, f0 * (1 - 0.15 * (i % 8) / 8), sr, r))
        parts.append(np.zeros(int(0.03 * sr / rate)))
    parts.append(np.zeros(int(0.3 * sr)))
    y = np.concatenate(parts)
    y = y / (np.abs(y).max() + 1e-9) * 0.5 + r.standard_normal(len(y)) * 1e-3
    return y.astype(np.float32)


# ---------- edge-tts 包裝：成功 → WAV；失敗 → 佔位聲音 ----------
try:
    import edge_tts
except ImportError:
    edge_tts = None
EDGE_STATE = {"online": edge_tts is not None}


async def tts_to_wav(text, wav_path, voice="zh-TW-HsiaoChenNeural", rate="+0%",
                     pitch="+0Hz", volume="+0%", label=None):
    """edge-tts 合成 → MP3 → WAV。回傳實際使用的引擎（"edge-tts" 或 "placeholder"）。"""
    label = label or Path(wav_path).stem
    engine = "placeholder"
    t0 = time.perf_counter()
    if EDGE_STATE["online"]:
        mp3 = Path(wav_path).with_suffix(".mp3")
        try:
            communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch, volume=volume)
            await asyncio.wait_for(communicate.save(str(mp3)), timeout=90)
            to_wav(mp3, wav_path)
            engine = "edge-tts"
        except Exception as err:
            network = (isinstance(err, (OSError, asyncio.TimeoutError))
                       or type(err).__module__.startswith("aiohttp")
                       or type(err).__name__ == "WebSocketError")
            if network:
                EDGE_STATE["online"] = False
                print(f"edge-tts 連線失敗（{type(err).__name__}）→ 之後一律改用離線佔位聲音。")
            else:
                print(f"edge-tts 合成失敗（{type(err).__name__}: {err}）→ 這一段改用佔位聲音。"
                      "（聲音名稱是否已變動？可執行 edge-tts --list-voices 確認）")
    if engine == "placeholder":
        pct = int(re.sub(r"[^-\d]", "", rate) or 0)
        y = placeholder_speech(text, sr=24000, seed=sum(map(ord, voice)) % 997, rate=1 + pct / 100)
        sf.write(str(wav_path), y, 24000, subtype="PCM_16")
    GEN_LOG[label] = dict(engine=engine, tool=f"edge-tts {pkg_version('edge-tts')}", voice=voice,
                          params=f"rate={rate}, pitch={pitch}, volume={volume}",
                          seconds=round(time.perf_counter() - t0, 2), path=str(wav_path),
                          sr=sf.info(str(wav_path)).samplerate, text=text, date=TODAY)
    STATUS[f"TTS {label}"] = "real" if engine == "edge-tts" else "fallback"
    return engine


print("工具函式就緒｜edge-tts", pkg_version("edge-tts"), "｜librosa", librosa.__version__)

### 0.3 選用：錄製或上傳自己的聲音

後面有三個地方會用到**你自己的聲音** `my_voice.wav`：

1. **12.14／練習 2**：Whisper 辨識真人錄音（練習 2 需要約 60 秒）。
2. **Z.6**：只用**本人**聲音做 voice cloning（另有同意開關）。
3. **練習 4**：證據卡比較「真實自錄」與「合成語音」。

建議朗讀 12.14 的稿子：

> 今天測試 Whisper 語音辨識。我們使用 Python、NumPy 與 librosa 分析聲音，STFT 的 Window Size 設定為 2048。

做法：把 `RECORD` 改成 `True`（Colab 會要求麥克風權限，錄 `RECORD_SECONDS` 秒），或把 `UPLOAD` 改成 `True` 上傳 wav／mp3／m4a。兩者預設都是 `False`，所以「全部執行」時不會停下來等你。

> **隱私提醒**：你的聲音檔就是可以拿來複製你聲音的素材。課程結束後請從 Colab 與雲端硬碟刪除，不要公開上傳。

In [ ]:
RECORD = False          # 改成 True：用瀏覽器麥克風錄音
UPLOAD = False          # 改成 True：上傳既有錄音檔
RECORD_SECONDS = 20     # 練習 2 請改成 60

RECORD_JS = """
async function recordAudio(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  const note = document.createElement("p");
  note.textContent = "錄音中…請開始朗讀";
  document.body.appendChild(note);
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await new Promise(r => setTimeout(r, ms));
  const stopped = new Promise(r => recorder.onstop = r);
  recorder.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  note.textContent = "錄音結束";
  const blob = new Blob(chunks);
  return await new Promise(r => {
    const fr = new FileReader();
    fr.onloadend = () => r(fr.result);
    fr.readAsDataURL(blob);
  });
}
"""


def record_colab(seconds, out_path="my_voice.wav", sr=44100):
    """在 Colab 用 JS MediaRecorder 錄音（webm/ogg）→ ffmpeg 轉成 WAV。"""
    from base64 import b64decode
    from google.colab import output
    from IPython.display import Javascript
    display(Javascript(RECORD_JS))
    data_url = output.eval_js(f"recordAudio({int(seconds * 1000)})")
    Path("my_voice_raw.webm").write_bytes(b64decode(data_url.split(",", 1)[1]))
    return to_wav("my_voice_raw.webm", out_path, sr=sr)


try:
    if RECORD:
        record_colab(RECORD_SECONDS)
    elif UPLOAD:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            to_wav(next(iter(uploaded)), "my_voice.wav", sr=44100)
except Exception as err:
    print(f"錄音／上傳失敗（{type(err).__name__}: {err}）。不在 Colab 時無法使用麥克風。")

if Path("my_voice.wav").exists():
    STATUS["本人錄音 my_voice.wav"] = "real"
    print(f"my_voice.wav：{duration('my_voice.wav'):.1f} 秒")
    play("my_voice.wav", label="my_voice.wav")
else:
    STATUS["本人錄音 my_voice.wav"] = "skipped"
    print("沒有 my_voice.wav：需要本人錄音的步驟會自動略過或改用合成語音。")

## 12.22 本週安全邊界（先讀）

本週會生成「像人」的聲音，也會用到**聲音複製**。動手之前，先把邊界講清楚：

| | 內容 |
|---|---|
| **允許** | 自己錄製的聲音、TTS 服務／模型的內建聲音、虛構角色、有授權的資料集 |
| **禁止** | 未經同意複製**可辨識的真人**聲音、冒充他人、把合成語音當成真實錄音發布、任何欺騙用途 |
| **本課額外規定** | 即使對方**口頭同意**，也**不得**複製第三方（同學、家人、老師、名人）的聲音 |

為什麼連「口頭同意」都不行？口頭同意無法查證、範圍不清楚（同意的是課堂練習，還是可以被放上網？）、也很難撤回；而聲音檔與複製模型一旦離開教室就收不回來。課堂上只用**你自己**的聲音，風險由你自己決定、也只落在你自己身上。

In [ ]:
SAFETY_BOUNDARY = """\
【W12 安全邊界】
允許：自己錄製的聲音、TTS 內建聲音、虛構角色、有授權的資料集。
禁止：未經同意複製可辨識真人的聲音、冒充他人、把合成語音當成真實錄音發布、用於欺騙。
本課額外規定：即使對方口頭同意，也不得複製第三方的聲音。"""
print(SAFETY_BOUNDARY)

## 12.1、12.3 TTS 在做什麼？

**概念**：TTS（text-to-speech）的**輸入是文字**，**輸出是波形**。傳統兩段式神經 TTS 的管線如下：

```
文字 ──► 文字分析 ──────────────► 聲學模型 ──► 聲學特徵（Mel 頻譜）──► vocoder ──► 波形
        正規化「3.5 秒」→「三點五秒」          80 列 × T 影格的矩陣      把 Mel 還原成
        斷詞、注音／音素、韻律                （12.2 會畫出來）          可播放的聲音
```

- **Mel 頻譜**就是 W10 頻譜圖的「壓縮版」：頻率軸改成接近人耳感受的 Mel 刻度、列數從上千列降到約 80 列（12.2）。
- **vocoder** 負責把 Mel 變回波形。Mel 丟掉了**相位**與細部頻率，所以還原不是查表，而是「猜」（12.1 用 Griffin-Lim 體驗）。
- 不是所有 TTS 都走 Mel：端到端模型直接輸出波形；ZONOS3-Mini 這類**語音編解碼器（codec）語言模型**則是預測 codec 的離散代碼（`codes`），再由 codec 解碼器還原波形。

**三種常被混為一談的技術**

| | 輸入 | 輸出 | 本週例子 |
|---|---|---|---|
| TTS | 文字（＋選一個內建聲音）| 該聲音念出文字 | edge-tts、ZONOS `voice="Meilin"` |
| 語者條件 TTS（zero-shot voice cloning）| 文字＋一段**參考聲音**（或它的語者嵌入）| 「聽起來像」參考聲音的人念出文字 | ZONOS `speaker_audio=`、`speaker_embedding=` |
| 語音轉換（voice conversion, VC）| 一段 A 說話的**錄音** | 內容、節奏不變，音色換成 B | （本週不實作）|

**12.3 語者嵌入（speaker embedding）**：把一段聲音壓成固定長度的向量（ZONOS3-Mini 是 192 維）。音色相近的聲音，向量方向也接近，**餘弦相似度**就高。它描述的是「聽起來像不像」，**不是身分證**——Z.5、Z.6 會看到複製聲音和本人的相似度可以很高。

## 12.4 edge-tts：第一段合成語音

**概念**：`edge-tts` 把文字送到 Microsoft Edge 的線上朗讀服務，取回 MP3。三個重點：

1. **需要網路**；它是**第三方**用戶端，不是官方 Azure Speech SDK，服務或聲音名稱隨時可能變動。
2. 聲音名稱的格式是「語言-地區-名字Neural」，例如 `zh-TW-HsiaoChenNeural`。
3. `Communicate(text, voice, rate=..., pitch=..., volume=...)` 可以調語速、音高、音量，例如 `rate="+20%"`。

先列出可用的臺灣華語聲音（命令列：`edge-tts --list-voices`）：

> **先預測再執行**：合成出來的檔案是什麼格式？取樣率大概多少？會和 W11 的 44.1 kHz 一樣嗎？

In [ ]:
!edge-tts --list-voices 2>/dev/null | grep -E "Name|zh-TW" || echo "無法取得聲音清單（可能沒有網路，或 edge-tts 未安裝）"

講義上的程式是 **Python 腳本**寫法，最後一行用 `asyncio.run()` 啟動非同步函式：

```python
import asyncio
import edge_tts

async def speak(text, out="speech.mp3", voice="zh-TW-HsiaoChenNeural"):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(out)

asyncio.run(speak("大家好，歡迎來到多媒體系統課程。"))
```

但在 **Colab／Jupyter** 裡，筆記本本身**已經有一個事件迴圈（event loop）在執行**，`asyncio.run()` 會想再開一個新的迴圈，於是報錯 `asyncio.run() cannot be called from a running event loop`。在筆記本中要改成**直接 `await`**：

```python
await speak("大家好，歡迎來到多媒體系統課程。")
```

（IPython 允許在格子最外層使用 `await`；一般 `.py` 檔不行，所以腳本才需要 `asyncio.run()`。）

In [ ]:
async def speak(text, out="speech.mp3", voice="zh-TW-HsiaoChenNeural"):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(out)


HELLO = "大家好，歡迎來到多媒體系統課程。"
try:
    if edge_tts is None:
        raise ImportError("edge-tts 未安裝")
    await asyncio.wait_for(speak(HELLO), timeout=90)   # Colab：直接 await，不用 asyncio.run()
    to_wav("speech.mp3", "speech.wav")
    STATUS["12.4 speak()"] = "real"
    print("已產生 speech.mp3，並轉成 speech.wav 方便分析")
except Exception as err:
    print(f"edge-tts 無法使用（{type(err).__name__}: {err}）→ 改用離線佔位聲音 speech.wav，讓後面流程能繼續。")
    EDGE_STATE["online"] = False
    sf.write("speech.wav", placeholder_speech(HELLO, seed=1), 24000, subtype="PCM_16")
    STATUS["12.4 speak()"] = "fallback"

GEN_LOG["speech"] = dict(engine="edge-tts" if STATUS["12.4 speak()"] == "real" else "placeholder",
                         tool=f"edge-tts {pkg_version('edge-tts')}", voice="zh-TW-HsiaoChenNeural",
                         params="rate=+0%, pitch=+0Hz, volume=+0%", seconds=None, path="speech.wav",
                         sr=sf.info("speech.wav").samplerate, text=HELLO, date=TODAY)
info = sf.info("speech.wav")
print(f"speech.wav：{info.duration:.2f} 秒，{info.samplerate} Hz，{info.channels} 聲道")
play("speech.wav", label="speak() 的輸出")

下面這格示範：在已經有事件迴圈的筆記本裡呼叫 `asyncio.run()` 會發生什麼事（預期會看到 `RuntimeError`）。

In [ ]:
coro = speak("測試 asyncio.run。", out="asyncio_test.mp3")
try:
    asyncio.run(coro)
    print("asyncio.run() 成功（代表目前沒有正在執行的事件迴圈，例如在一般 Python 腳本中）")
except RuntimeError as err:
    print("RuntimeError：", err)
    print("→ 這就是在 Colab 要改用 `await speak(...)` 的原因。")
except Exception as err:
    print(f"其他錯誤（{type(err).__name__}）：{err}")
finally:
    coro.close()   # 沒被執行的 coroutine 要關掉，避免 "was never awaited" 警告

講義的腳本寫法其實沒錯，只是要在**筆記本以外**執行。下面用 `%%writefile` 把它原樣存成 `tts_demo.py`，再用 `!python` 在另一個行程中執行——那裡沒有事件迴圈，`asyncio.run()` 就能正常運作。

In [ ]:
%%writefile tts_demo.py
import asyncio
import edge_tts

async def speak(text, out="speech.mp3", voice="zh-TW-HsiaoChenNeural"):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(out)

asyncio.run(speak("大家好，歡迎來到多媒體系統課程。"))

In [ ]:
!timeout 120 python tts_demo.py > tts_demo.log 2>&1 && echo "腳本執行成功：speech.mp3 已更新" || (echo "腳本執行失敗（多半是沒有網路），最後幾行訊息："; tail -n 2 tts_demo.log)

### 換聲音、調語速

同一句話換三個臺灣華語聲音，再把語速調快 20%：

```python
edge_tts.Communicate(text, "zh-TW-HsiaoChenNeural", rate="+20%")
```

> **先預測再執行**：`rate="+20%"` 之後，長度會變成原本的幾倍？音高會變嗎？（對照 W11 的 time stretch。）

In [ ]:
VOICES = ["zh-TW-HsiaoChenNeural", "zh-TW-YunJheNeural", "zh-TW-HsiaoYuNeural"]
rows = []
for v in VOICES:
    name = "voice_" + v.split("-")[2].replace("Neural", "")
    eng = await tts_to_wav(HELLO, f"{name}.wav", voice=v)
    rows.append((name, v, "+0%", eng, round(duration(f"{name}.wav"), 2), sf.info(f"{name}.wav").samplerate))
    play(f"{name}.wav", label=f"{v}（{eng}）")

eng = await tts_to_wav(HELLO, "rate_plus20.wav", voice=VOICES[0], rate="+20%")
rows.append(("rate_plus20", VOICES[0], "+20%", eng, round(duration("rate_plus20.wav"), 2),
             sf.info("rate_plus20.wav").samplerate))
play("rate_plus20.wav", label=f"{VOICES[0]}，rate=+20%（{eng}）")
show_table(["檔案", "voice", "rate", "引擎", "長度 (s)", "取樣率 (Hz)"], rows)

base = [r for r in rows if r[0] == "voice_HsiaoChen"][0][4]
print(f"長度比 rate+20% / 原速 = {rows[-1][4] / base:.2f}（理論上約 1/1.2 = {1 / 1.2:.2f}）")

**觀察**：edge-tts 預設輸出 **24 kHz 單聲道 MP3**，所以 12 kHz 以上沒有任何內容（奈奎斯特頻率，W10）；ZONOS3-Mini 則輸出 44.1 kHz。這個「頻寬」在 12.16 的證據卡會變成一條**可觀察線索**——但電話錄音一樣是窄頻，所以只是線索，不是證據。

**聆聽檢核**
- [ ] 三個聲音的音色、性別印象、說話節奏不同。
- [ ] `rate="+20%"` 變快、變短，但音高沒有變高（不是花栗鼠聲）。
- [ ] 若看到「引擎 = placeholder」，代表 edge-tts 沒有連上，聽到的只是佔位聲音。

## 12.2 Mel 頻譜：TTS 的「中間產物」

**概念**：W10 的 STFT 頻譜圖，頻率軸是**線性**的：每個頻率格相隔 `sr / n_fft` Hz。人耳對低頻的變化很敏感、對高頻較遲鈍，所以 TTS 改用 **Mel 刻度**：

  m = 2595 · log10(1 + f / 700)

- 約 1 kHz 以下近似線性、以上近似對數。
- `librosa.feature.melspectrogram` 先算 STFT 的功率譜，再乘上一組三角形的 **Mel 濾波器組**（`librosa.filters.mel`），把上千個頻率格「合併」成 80 個 Mel 頻帶。
- 注意：上面的公式是 **HTK** 版本（`htk=True`）；librosa 預設是 **Slaney** 版本（1 kHz 以下嚴格線性），兩者形狀接近但數值不同——寫報告時要說明用哪一種。

> **先預測再執行**：`n_fft=1024` 時線性 STFT 有幾列？`n_mels=80` 的 Mel 頻譜有幾列？壓縮了幾倍？

In [ ]:
y_tts, sr_tts = load_audio("speech.wav")
N_FFT, HOP, N_MELS = 1024, 256, 80

S_lin = np.abs(librosa.stft(y_tts, n_fft=N_FFT, hop_length=HOP))
M = librosa.feature.melspectrogram(y=y_tts, sr=sr_tts, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
print("線性 STFT 形狀（頻率格, 影格）:", S_lin.shape)
print("Mel 頻譜形狀（Mel 頻帶, 影格）:", M.shape, f"→ 列數壓縮 {S_lin.shape[0] / M.shape[0]:.1f} 倍")
print(f"每個線性頻率格寬 {sr_tts / N_FFT:.1f} Hz；每個影格 {HOP / sr_tts * 1000:.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
img = librosa.display.specshow(librosa.amplitude_to_db(S_lin, ref=np.max), sr=sr_tts, hop_length=HOP,
                               x_axis="time", y_axis="linear", ax=axes[0])
axes[0].set_title(f"Linear STFT ({S_lin.shape[0]} bins)")
fig.colorbar(img, ax=axes[0], format="%+.0f dB")
img = librosa.display.specshow(librosa.power_to_db(M, ref=np.max), sr=sr_tts, hop_length=HOP,
                               x_axis="time", y_axis="mel", ax=axes[1])
axes[1].set_title(f"Mel spectrogram ({N_MELS} bands)")
fig.colorbar(img, ax=axes[1], format="%+.0f dB")
plt.tight_layout()
plt.show()

In [ ]:
f = np.linspace(0, sr_tts / 2, 500)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(f, 2595 * np.log10(1 + f / 700), label="2595*log10(1+f/700) (HTK)")
axes[0].plot(f, librosa.hz_to_mel(f, htk=False) * (2595 * np.log10(1 + 4000 / 700)) / librosa.hz_to_mel(4000),
             "--", label="librosa default (Slaney), rescaled")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Mel")
axes[0].set_title("Mel scale")
axes[0].legend()

fb = librosa.filters.mel(sr=sr_tts, n_fft=N_FFT, n_mels=40)
freqs = librosa.fft_frequencies(sr=sr_tts, n_fft=N_FFT)
for row in fb[::3]:
    axes[1].plot(freqs, row)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Weight")
axes[1].set_title("Mel filterbank (every 3rd of 40 filters)")
plt.tight_layout()
plt.show()

print("驗算 1000 Hz：公式 =", round(2595 * np.log10(1 + 1000 / 700), 2),
      "｜librosa.hz_to_mel(1000, htk=True) =", round(float(librosa.hz_to_mel(1000, htk=True)), 2))
print("濾波器組形狀（Mel 頻帶, 線性頻率格）:", fb.shape)

**觀察**：低頻的三角形濾波器又窄又密、高頻的又寬又疏——高頻細節被「合併」得最多。Mel 頻譜上，語音的諧波（W10 看到的一條條橫紋）在低頻仍清楚，高頻則糊成一片。

**檢核**
- [ ] 我能說出 Mel 頻譜和 W10 頻譜圖的相同點（都是 STFT 的幅度、都是時間 × 頻率）與不同點（頻率刻度、列數、只保留功率）。
- [ ] 我知道濾波器組矩陣的形狀是 (n_mels, 1 + n_fft/2)。

## 12.1 vocoder 的直覺：從 Mel 還原聲音

**概念**：Mel 頻譜只保留「每個頻帶有多少能量」，**相位全部丟掉了**，頻率細節也被合併。要變回波形，必須「猜」出相位：

- **隨機相位**：直接配上亂數相位做 ISTFT——能量分佈對，但聽起來像沙沙的耳語。
- **Griffin-Lim**：反覆做 STFT／ISTFT，讓相位和幅度彼此一致（`librosa.feature.inverse.mel_to_audio` 預設迭代 32 次）——好很多，但仍有金屬感、嗡嗡聲。
- **神經 vocoder**（WaveNet、HiFi-GAN 等）：從大量真實語音**學會**「什麼樣的波形會產生這個 Mel」，所以能補出自然的相位與細節。這就是現代 TTS 需要神經 vocoder 的原因。

> **先預測再執行**：Griffin-Lim 迭代 8 次和 64 次，哪個比較像原聲？差別大嗎？

In [ ]:
S_from_mel = librosa.feature.inverse.mel_to_stft(M, sr=sr_tts, n_fft=N_FFT)        # 只有幅度
rand_phase = np.exp(2j * np.pi * rng.random(S_from_mel.shape))
y_rand = librosa.istft(S_from_mel * rand_phase, hop_length=HOP, length=len(y_tts))

recon = {"Original": y_tts, "Mel + random phase": y_rand}
for n_iter in [8, 64]:
    t0 = time.perf_counter()
    recon[f"Griffin-Lim n_iter={n_iter}"] = librosa.feature.inverse.mel_to_audio(
        M, sr=sr_tts, n_fft=N_FFT, hop_length=HOP, n_iter=n_iter, length=len(y_tts))
    print(f"Griffin-Lim n_iter={n_iter}: {time.perf_counter() - t0:.2f} s")


def spectral_convergence(y_ref, y_est):
    """幅度譜相對誤差：越小越接近（只比幅度，不比相位）。"""
    A = np.abs(librosa.stft(y_ref, n_fft=N_FFT, hop_length=HOP))
    B = np.abs(librosa.stft(y_est, n_fft=N_FFT, hop_length=HOP))
    return float(np.linalg.norm(A - B) / (np.linalg.norm(A) + 1e-9))


show_table(["版本", "幅度譜相對誤差"],
           [(k, round(spectral_convergence(y_tts, v), 3)) for k, v in recon.items()])

fig, axes = plt.subplots(1, len(recon), figsize=(14, 3), sharey=True)
for ax, (k, v) in zip(axes, recon.items()):
    librosa.display.specshow(librosa.amplitude_to_db(np.abs(librosa.stft(v, n_fft=N_FFT, hop_length=HOP)), ref=np.max),
                             sr=sr_tts, hop_length=HOP, x_axis="time", y_axis="linear", ax=ax)
    ax.set_title(k, fontsize=9)
plt.tight_layout()
plt.show()
for k, v in recon.items():
    play(v / (np.abs(v).max() + 1e-9) * 0.5, sr_tts, label=k)

**聆聽檢核**
- [ ] 隨機相位版本「字」大致還在，但像耳語、沒有清楚的音高。
- [ ] Griffin-Lim 比隨機相位好，迭代越多越好，但仍有金屬感或「機器人」感。
- [ ] 我能用一句話說明 vocoder 的角色：**把聲學特徵（Mel）還原成波形，並補回 Mel 丟掉的相位與細節**。

## 12.5 控制變因盲測

**概念**：「哪個 TTS 比較自然？」不能只憑感覺，要像實驗一樣設計：

1. **固定**文字（同一句話）、播放音量與播放裝置。
2. **只改一個變因**：要嘛只換 `voice`，要嘛只改 `rate`，不要同時改。
3. **打亂代號**：把版本隨機標成 A／B／C（用固定 `seed`，之後可重現），聽者不知道哪個是哪個。
4. **先評分、後揭曉**：每位聽者對每個版本打三項 1–5 分：

| 分數 | 自然度（像不像真人）| 清晰度（聽不聽得懂）| 韻律（停頓、重音、語調）|
|---|---|---|---|
| 5 | 幾乎聽不出是合成 | 每個字都清楚 | 停頓與語調都自然 |
| 3 | 明顯是合成但可接受 | 偶爾要猜 | 有幾處怪怪的 |
| 1 | 非常機械 | 很多字聽不懂 | 斷句錯誤、語調平板 |

下面的示範以 **`rate`** 為變因（同一個聲音，−20%／+0%／+20%）。想改以聲音為變因，把 `VARIABLE` 改成 `"voice"`。

In [ ]:
BLIND = {}   # 盲測名稱 → 代號對照（揭曉前不要印出來）


def make_blind_test(name, items, seed):
    """items：[(說明, wav 路徑), ...]。打亂後以 A/B/C… 播放，對照表存到 BLIND[name] 與 json 檔。"""
    order = list(range(len(items)))
    random.Random(seed).shuffle(order)
    key = {}
    for letter, idx in zip("ABCDEFGH", order):
        desc, path = items[idx]
        key[letter] = {"setting": desc, "path": str(path)}
        play(path, label=f"版本 {letter}")
    BLIND[name] = key
    Path(f"blind_key_{name}.json").write_text(json.dumps(key, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"共 {len(key)} 個版本，對照表已存成 blind_key_{name}.json（評分完成前不要打開）。")


def reveal(name, scores):
    """揭曉代號，並計算每個版本三項分數的平均（未填的分數會略過）。"""
    rows = []
    for letter, info in BLIND[name].items():
        got = [s[letter] for s in scores.values() if s.get(letter)]
        avg = [round(float(np.mean([g[i] for g in got])), 2) if got else "—" for i in range(3)]
        rows.append((letter, info["setting"], GEN_LOG.get(Path(info["path"]).stem, {}).get("engine", "?"),
                     len(got), *avg))
    show_table(["代號", "實際設定", "引擎", "評分人數", "自然度", "清晰度", "韻律"], rows)


BLIND_TEXT = "今天的多媒體系統課程，我們要比較三種語音合成的設定，請仔細聆聽每一個版本。"
BLIND_SEED = 1205
VARIABLE = "rate"    # "rate" 或 "voice"：一次只改一個
CONDITIONS = {
    "rate": [dict(voice="zh-TW-HsiaoChenNeural", rate=r) for r in ["-20%", "+0%", "+20%"]],
    "voice": [dict(voice=v, rate="+0%") for v in VOICES],
}[VARIABLE]

items = []
for i, cond in enumerate(CONDITIONS):
    path = f"blind_edge_{i}.wav"
    await tts_to_wav(BLIND_TEXT, path, **cond)
    items.append((f"{cond['voice']}, rate={cond['rate']}", path))
print(f"固定文字：{BLIND_TEXT}\n變因：{VARIABLE}\n")
make_blind_test("edge", items, seed=BLIND_SEED)

**評分表**（先填這張，再執行下一格揭曉）

| 聽者 | 版本 | 自然度 1–5 | 清晰度 1–5 | 韻律 1–5 | 一句描述 |
|---|---|---|---|---|---|
| 聽者 1 | A | | | | |
| 聽者 1 | B | | | | |
| 聽者 1 | C | | | | |
| 聽者 2 | A | | | | |
| 聽者 2 | B | | | | |
| 聽者 2 | C | | | | |

In [ ]:
# TODO：把評分填進來，格式是 (自然度, 清晰度, 韻律)；沒填的保持 None
SCORES_EDGE = {
    "listener_1": {"A": None, "B": None, "C": None},
    "listener_2": {"A": None, "B": None, "C": None},
}
reveal("edge", SCORES_EDGE)

**檢核**
- [ ] 我在揭曉之前就完成評分，而且至少兩位聽者。
- [ ] 我只改了一個變因；結論只談這個變因（例如「語速 +20% 清晰度下降」），不推論到其他設定。
- [ ] 若引擎顯示 `placeholder`，這次盲測不能當作結論，要在有網路的環境重做。

## 12.6 Whisper：在 Colab 本地做語音辨識

**概念**：Whisper 是 OpenAI 開源的語音辨識模型。`pip install -U openai-whisper` 之後，模型權重會下載到 Colab 主機，**音檔在本地辨識、不必上傳到第三方 API**——這是隱私上的好處（但權重下載仍需網路，而 Colab 主機本身也是雲端，敏感錄音仍要小心）。Whisper 讀檔時會呼叫 **FFmpeg**，Colab 已內建。

| 模型 | 參數量（約）| 特性 |
|---|---|---|
| `tiny` | 39 M | 最快，適合確認環境 |
| `base` | 74 M | |
| `small` | 244 M | 本週正式比較用 |
| `medium` | 769 M | T4 可跑，較慢 |
| `large` | 1.55 B | 需要較多 GPU 記憶體 |

先準備本週的測試音檔與參考稿：

- `reference.txt`：12.14 的朗讀稿（標準答案）。
- `lecture.wav`：如果你在 0.3 錄了 `my_voice.wav`（朗讀這份稿子），就用你的錄音；否則用 edge-tts 念這份稿子（`script_tts.wav`）。

In [ ]:
REFERENCE = "今天測試 Whisper 語音辨識。我們使用 Python、NumPy 與 librosa 分析聲音，STFT 的 Window Size 設定為 2048。"
Path("reference.txt").write_text(REFERENCE + "\n", encoding="utf-8")

await tts_to_wav(REFERENCE, "script_tts.wav", voice="zh-TW-HsiaoChenNeural", label="script_tts")

USE_MY_VOICE_FOR_LECTURE = Path("my_voice.wav").exists()   # 若你的錄音不是朗讀這份稿子，改成 False
LECTURE_SOURCE = "my_voice.wav" if USE_MY_VOICE_FOR_LECTURE else "script_tts.wav"
to_wav(LECTURE_SOURCE, "lecture.wav", sr=16000)            # Whisper 內部本來就用 16 kHz
LECTURE_ENGINE = "本人錄音" if USE_MY_VOICE_FOR_LECTURE else GEN_LOG["script_tts"]["engine"]
print(f"lecture.wav ← {LECTURE_SOURCE}（{LECTURE_ENGINE}），{duration('lecture.wav'):.2f} 秒")
if LECTURE_ENGINE == "placeholder":
    print("注意：lecture.wav 是佔位聲音，Whisper 的辨識結果沒有意義，只用來確認流程。")
play("lecture.wav", label="lecture.wav")

講義的最小範例：先用 `tiny` 確認環境，再看完整文字與每一段的時間戳。

> **先預測再執行**：`tiny` 會把「NumPy」「librosa」「STFT」寫對嗎？輸出會是繁體還是簡體？

在 CPU 上會看到 `FP16 is not supported on CPU; using FP32 instead` 的警告，可以忽略。

In [ ]:
WHISPER_CACHE = {}                    # 模型名稱 → 已載入的模型（載入失敗則為 None，避免重試）
WHISPER_STATE = {"importable": None}


def get_whisper(name):
    if name in WHISPER_CACHE:
        return WHISPER_CACHE[name]
    if WHISPER_STATE["importable"] is False:
        return None
    try:
        import whisper
        WHISPER_STATE["importable"] = True
    except Exception as err:
        WHISPER_STATE["importable"] = False
        print(f"無法匯入 whisper（{type(err).__name__}: {err}）→ 需要 Whisper 的步驟會略過。")
        return None
    try:
        t0 = time.perf_counter()
        m = whisper.load_model(name, device=DEVICE)
        print(f"載入 Whisper {name}：{time.perf_counter() - t0:.1f} s（{DEVICE}）")
        STATUS[f"Whisper {name}"] = "real"
    except Exception as err:
        print(f"載入 Whisper {name} 失敗（{type(err).__name__}: {err}）。下載權重需要網路。")
        STATUS[f"Whisper {name}"] = "fallback"
        m = None
    WHISPER_CACHE[name] = m
    return m


def transcribe(path, name="tiny", **kwargs):
    """回傳 (result, 轉錄秒數)；模型不可用時回傳 (None, None)。"""
    m = get_whisper(name)
    if m is None:
        return None, None
    kwargs.setdefault("language", "zh")
    kwargs.setdefault("fp16", DEVICE == "cuda")
    t0 = time.perf_counter()
    res = m.transcribe(str(path), **kwargs)
    return res, time.perf_counter() - t0


result = None
try:
    import whisper

    t0 = time.perf_counter()
    model = whisper.load_model("tiny")
    print(f"載入 tiny：{time.perf_counter() - t0:.1f} s｜裝置：{DEVICE}（{GPU_NAME}）")
    WHISPER_CACHE["tiny"] = model
    STATUS["Whisper tiny"] = "real"

    t0 = time.perf_counter()
    result = model.transcribe("lecture.wav", language="zh")
    print(f"轉錄時間：{time.perf_counter() - t0:.2f} s（音檔 {duration('lecture.wav'):.1f} s）\n")

    print(result["text"])
    for seg in result["segments"]:
        print(f"[{seg['start']:7.2f} → {seg['end']:7.2f}] {seg['text']}")
except Exception as err:
    print(f"Whisper 無法執行（{type(err).__name__}: {err}）。")
    print("→ 可能原因：openai-whisper 未安裝、沒有網路下載權重。後面需要 Whisper 的格子會略過或用示範文字。")
    STATUS.setdefault("Whisper tiny", "fallback")
    WHISPER_CACHE.setdefault("tiny", None)     # 之後的 get_whisper("tiny") 不再重試下載
    if isinstance(err, ImportError):
        WHISPER_STATE["importable"] = False

**觀察**：`result` 是一個字典：`result["text"]` 是整段文字，`result["segments"]` 是一段一段的結果，每段有 `start`、`end`（秒）與 `text`。

## 12.9 時間戳只是估計

**概念**：Whisper 的時間戳是模型**預測**出來的（以 0.02 秒為單位的時間標記），不是對齊演算法量出來的：段落邊界可能提早或延後、常常把停頓算進段落、偶爾整段漂移。拿來做字幕前，一定要**用耳朵和波形檢查**。

下面把 Whisper 的段落和 W11 的「能量式語音偵測」（`librosa.effects.split`，相當於 `detect_nonsilent`）畫在一起比較。

> **先預測再執行**：Whisper 段落的起點，會比能量偵測到的語音起點早還是晚？

In [ ]:
DEMO_SEGMENTS = [   # Whisper 不可用時的示範資料（不是真的辨識結果）
    {"start": 0.00, "end": 2.40, "text": "今天測試 Whisper 語音辨識。"},
    {"start": 2.40, "end": 7.86, "text": "我們使用 Python、NumPy 與 librosa 分析聲音，"},
    {"start": 7.86, "end": 11.20, "text": "STFT 的 Window Size 設定為 2048。"},
]
segments = result["segments"] if result else DEMO_SEGMENTS
if not result:
    print("（Whisper 未執行：以下使用示範段落 DEMO_SEGMENTS）")

y_lec, sr_lec = load_audio("lecture.wav")
voiced = librosa.effects.split(y_lec, top_db=35) / sr_lec       # 能量式語音段（秒）
t = np.arange(len(y_lec)) / sr_lec

fig, axes = plt.subplots(2, 1, figsize=(11, 4.4), sharex=True)
for ax, spans, title, color in [
    (axes[0], [(s["start"], s["end"]) for s in segments], "Whisper segments (predicted timestamps)", "tab:blue"),
    (axes[1], voiced, "Energy-based voiced spans (librosa.effects.split, top_db=35)", "tab:orange"),
]:
    ax.plot(t, y_lec, lw=0.4, color="0.3")
    for k, (a, b) in enumerate(spans):
        ax.axvspan(a, b, color=color, alpha=0.25 if k % 2 == 0 else 0.45)
    ax.set_title(title)
    ax.set_ylabel("Amplitude")
axes[1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

rows = []
for s in segments:
    nearest = min(voiced[:, 0], key=lambda v: abs(v - s["start"])) if len(voiced) else float("nan")
    rows.append((f"{s['start']:.2f}", f"{s['end']:.2f}", f"{nearest:.2f}", f"{s['start'] - nearest:+.2f}", s["text"].strip()))
show_table(["Whisper start", "Whisper end", "最近的能量起點", "差 (s)", "文字"], rows)

### 延伸：把段落寫成 SRT 字幕

SRT 的時間格式是 `時:分:秒,毫秒`，例如 1.23 秒 → `00:00:01,230`（注意是**逗號**）。每一則字幕是「編號、時間、文字、空行」。

In [ ]:
def srt_time(t):
    """秒 → SRT 時間字串 00:00:01,230。"""
    ms = int(round(max(float(t), 0.0) * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def segments_to_srt(segments, path=None, convert=None):
    """Whisper segments → SRT 文字；convert 可放 s2twp 等轉換函式（要在報告中宣告）。"""
    blocks = []
    for i, seg in enumerate(segments, start=1):
        text = seg["text"].strip()
        if convert:
            text = convert(text)
        blocks.append(f"{i}\n{srt_time(seg['start'])} --> {srt_time(seg['end'])}\n{text}\n")
    srt = "\n".join(blocks)
    if path:
        Path(path).write_text(srt, encoding="utf-8")
    return srt


print("srt_time(1.23) =", srt_time(1.23))
print("srt_time(3725.5) =", srt_time(3725.5))
print(segments_to_srt(segments, "lecture.srt"))
print("已寫入 lecture.srt" + ("" if result else "（示範段落）"))

**檢核**
- [ ] 我能指出至少一個 Whisper 段落邊界和波形不一致的地方（或說明為什麼這次剛好一致）。
- [ ] 我知道 SRT 的毫秒前面是逗號，而且時間戳要人工檢查。

## 12.10 `initial_prompt` 與簡繁轉換

**概念**：`initial_prompt` 是一段「假裝是前文」的文字，會影響 Whisper 的**用字與風格**：提示中出現的專有名詞比較容易被寫對，提示用繁體也比較可能輸出繁體。但它**只是提示**：

- 不能保證每個詞都寫對，也不能修正聽錯的音。
- 太長會被截斷；提示中的詞偶爾會被「幻覺」寫進結果。
- 換一段錄音，效果可能就不同——所以要和「沒有提示」的結果並排比較。

> **先預測再執行**：加上提示後，「NumPy」「librosa」「STFT」的拼法會改善嗎？輸出會從簡體變繁體嗎？

In [ ]:
WHISPER_PROMPT = (
    "以下為繁體中文資訊課程內容，"
    "包含 NumPy、librosa、STFT、"
    "Whisper、OpenCV 等專有名詞。"
)
result_prompt = None
model = get_whisper("tiny")
if model is None:
    print("Whisper 未執行，略過 initial_prompt 比較。")
else:
    result_prompt = model.transcribe(
        "lecture.wav",
        language="zh",
        initial_prompt=(
            "以下為繁體中文資訊課程內容，"
            "包含 NumPy、librosa、STFT、"
            "Whisper、OpenCV 等專有名詞。"
        ),
        fp16=(DEVICE == "cuda"),
    )
    show_table(["設定", "辨識結果"], [("沒有 initial_prompt", result["text"] if result else "—"),
                                       ("有 initial_prompt", result_prompt["text"])])

### 簡體轉繁體：轉換也是管線的一部分

Whisper 的中文輸出常常是**簡體**。可以用 OpenCC 的 `s2twp`（簡體 → 臺灣正體，**含慣用詞轉換**，例如「软件」→「軟體」）處理：

```bash
pip install opencc-python-reimplemented
```

注意：`s2twp` 不只換字形，還會**換詞**。轉換之後再算 CER，結果會改變；所以「有沒有做簡轉繁、用哪一種轉換」必須寫進報告的正規化規則，不能偷偷做。

In [ ]:
try:
    from opencc import OpenCC
    try:
        CC = OpenCC("s2twp")
    except Exception:
        CC = OpenCC("s2twp.json")
    STATUS["OpenCC s2twp"] = "real"
except Exception as err:
    CC = None
    STATUS["OpenCC s2twp"] = "skipped"
    print(f"OpenCC 無法使用（{type(err).__name__}）：s2twp() 將原樣回傳。")


def s2twp(text):
    """簡體 → 臺灣正體（含慣用詞）。OpenCC 不可用時原樣回傳。"""
    return CC.convert(text) if CC else text


SIMPLIFIED_DEMO = "今天测试 Whisper 语音识别。我们使用 Python、NumPy 与 librosa 分析声音，STFT 的窗口大小设置为 2048。"
sample = (result_prompt or result or {}).get("text") or SIMPLIFIED_DEMO
show_table(["", "文字"], [("原始", sample), ("s2twp 之後", s2twp(sample)), ("示範（簡體）", SIMPLIFIED_DEMO),
                          ("示範 s2twp", s2twp(SIMPLIFIED_DEMO))])

**觀察**：示範句中「识别」會變成「識別」、「设置」可能變成「設定」——後者是**換詞**，不只是換字形。參考稿寫的是「辨識」，轉換後仍然不同，CER 依然會算它錯：簡轉繁不會修正**用詞**差異。

## 12.11–12.13 WER 與 CER：自己實作 Levenshtein 對齊

**概念**：把辨識結果（hypothesis）和參考稿（reference）**對齊**，數出三種錯誤：

| 錯誤 | 意思 | 例子（參考 → 辨識）|
|---|---|---|
| S 替換 substitution | 一個 token 被換成另一個 | 音 → 言 |
| D 刪除 deletion | 參考有、辨識漏掉 | 學習 → 學 |
| I 插入 insertion | 參考沒有、辨識多出來 | 語音 → 語音啊 |

- **WER** = (S + D + I) / N，N 是參考的**詞**數（英文以空白分詞）。
- **CER** = (S + D + I) / N_char，N_char 是參考的**字元**數。中文沒有空白分詞，「詞」的切法又因工具而異，所以中文通常用 **CER**。
- 錯誤率可以**超過 100%**（插入很多時）。

對齊用的是動態規劃的 **Levenshtein 編輯距離**：`d[i][j]` = 參考前 i 個 token 變成辨識前 j 個 token 的最少編輯次數；再從右下角回溯，就知道每一步是 =、S、D 還是 I。

> **先預測再執行**：5 個英文字錯 1 個，WER 是多少？「今天學習語音」被辨識成「今天學習語言」，CER 是多少？

In [ ]:
def align(ref, hyp):
    """Levenshtein 對齊。ref、hyp 是 token 串列（字或詞）。
    回傳 dict(S, D, I, H, N, ops)；ops 每列為 (操作, 參考 token, 辨識 token)，
    操作：'=' 正確、'S' 替換、'D' 刪除、'I' 插入。"""
    ref, hyp = list(ref), list(hyp)
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j - 1] + cost,   # 正確或替換
                          d[i - 1][j] + 1,          # 刪除
                          d[i][j - 1] + 1)          # 插入
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("=" if ref[i - 1] == hyp[j - 1] else "S", ref[i - 1], hyp[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(("D", ref[i - 1], ""))
            i -= 1
        else:
            ops.append(("I", "", hyp[j - 1]))
            j -= 1
    ops.reverse()
    count = {k: sum(op == k for op, _, _ in ops) for k in "=SDI"}
    assert count["S"] + count["D"] + count["I"] == d[n][m]
    return dict(S=count["S"], D=count["D"], I=count["I"], H=count["="], N=n, ops=ops)


def error_rate(res):
    """(S + D + I) / N；參考為空時回傳 nan。"""
    return (res["S"] + res["D"] + res["I"]) / res["N"] if res["N"] else float("nan")


def show_alignment(res, width=24):
    """把對齊結果畫成表格：REF／HYP／OP 三列（∅ 表示空）。"""
    ops = res["ops"]
    for k in range(0, len(ops), width):
        chunk = ops[k:k + width]
        esc = lambda s: (s or "∅").replace("|", "\\|")
        lines = ["| | " + " | ".join(str(k + i + 1) for i in range(len(chunk))) + " |",
                 "|---|" + "---|" * len(chunk),
                 "| REF | " + " | ".join(esc(r) for _, r, _ in chunk) + " |",
                 "| HYP | " + " | ".join(esc(h) for _, _, h in chunk) + " |",
                 "| OP | " + " | ".join("**" + op + "**" if op != "=" else "=" for op, _, _ in chunk) + " |"]
        display(Markdown("\n".join(lines)))
    print(f"S={res['S']}  D={res['D']}  I={res['I']}  正確={res['H']}  N={res['N']}  "
          f"錯誤率=({res['S']}+{res['D']}+{res['I']})/{res['N']} = {error_rate(res):.1%}")


def wer(ref, hyp, rules=()):
    return error_rate(align(normalize(ref, rules).split(), normalize(hyp, rules).split()))


def cer(ref, hyp, rules=()):
    return error_rate(align(list(normalize(ref, rules)), list(normalize(hyp, rules))))


def normalize(text, rules):
    for f in rules:
        text = f(text)
    return text


# 講義範例 1：5 個英文字、1 個替換 → WER = 1/5 = 20%
r1 = align("the cat sat on the".split(), "the cat sit on the".split())
show_alignment(r1)
assert (r1["S"], r1["D"], r1["I"], r1["N"]) == (1, 0, 0, 5) and abs(error_rate(r1) - 0.20) < 1e-12

# 講義範例 2：今天學習語音 → 今天學習語言，CER = 1/6 ≈ 16.7%
r2 = align(list("今天學習語音"), list("今天學習語言"))
show_alignment(r2)
assert (r2["S"], r2["N"]) == (1, 6) and abs(error_rate(r2) - 1 / 6) < 1e-12

# 刪除與插入都有的例子
show_alignment(align(list("我們使用語音辨識"), list("我使用語音辨識啊")))
print("講義範例驗算通過")

### 正規化規則：同一對句子，可以算出不同的 CER

**概念**：計分前要先決定「哪些差異不算錯」。常見規則：

| 規則 | 函式 | 例子 |
|---|---|---|
| 去除空白 | `rm_space` | 「Whisper 語音」→「Whisper語音」 |
| 去除標點（全形＋半形）| `rm_punct` | 「，。、！?,.」都移除 |
| 全形／半形統一（NFKC）| `to_halfwidth` | 「ＮｕｍＰｙ２０４８」→「NumPy2048」、「，」→「,」 |
| 英文小寫 | `lower_en` | 「STFT」→「stft」 |
| 簡轉繁（選用）| `s2twp` | 「语音识别」→「語音識別」 |

規則沒有對錯，但**一定要事先宣告、所有模型用同一套**，否則 CER 無法比較。

In [ ]:
def rm_space(s):
    return re.sub(r"\s+", "", s)


def rm_punct(s):
    """移除所有 Unicode 標點（類別 P*），全形與半形都包含。"""
    return "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))


def to_halfwidth(s):
    """NFKC：全形英數與全形標點轉成半形。"""
    return unicodedata.normalize("NFKC", s)


def lower_en(s):
    return s.lower()


RULESETS = {
    "R0 原樣": [],
    "R1 去空白": [rm_space],
    "R2 去空白＋去標點": [rm_space, rm_punct],
    "R3 NFKC＋小寫＋去標點＋去空白": [to_halfwidth, lower_en, rm_punct, rm_space],
    "R4 R3＋簡轉繁 s2twp": [s2twp, to_halfwidth, lower_en, rm_punct, rm_space],
}
SCORE_RULE = "R4 R3＋簡轉繁 s2twp"     # 本週預設的計分規則（報告中要寫出來）

PAIR_REF = "今天測試 Whisper 語音辨識，STFT 的 Window Size 設定為 2048。"
PAIR_HYP = "今天测试whisper语音识别,ＳＴＦＴ的window size设定为２０４８"
print("REF:", PAIR_REF)
print("HYP:", PAIR_HYP)
rows = []
for name, rules in RULESETS.items():
    rr = align(list(normalize(PAIR_REF, rules)), list(normalize(PAIR_HYP, rules)))
    rows.append((name, normalize(PAIR_HYP, rules), rr["N"], rr["S"], rr["D"], rr["I"], f"{error_rate(rr):.1%}"))
show_table(["規則", "正規化後的 HYP", "N_char", "S", "D", "I", "CER"], rows)
show_alignment(align(list(normalize(PAIR_REF, RULESETS[SCORE_RULE])), list(normalize(PAIR_HYP, RULESETS[SCORE_RULE]))))

**觀察**：同一對句子，CER 會隨規則大幅改變。即使用了 R4，「辨識」vs「識別」仍算錯——那是**用詞**差異，不是字形差異。

### 交叉驗證：和 `jiwer` 比較

自己寫的程式要驗證。`jiwer` 是常用的 WER／CER 套件；同樣的 token 串，**總錯誤數**必須一致（S／D／I 的分配在平手時可能不同，那是回溯規則的差異）。

In [ ]:
try:
    import jiwer
    checks = [("the cat sat on the", "the cat sit on the"),
              ("今天學習語音", "今天學習語言"),
              (normalize(PAIR_REF, RULESETS[SCORE_RULE]), normalize(PAIR_HYP, RULESETS[SCORE_RULE])),
              ("我們使用語音辨識", "我使用語音辨識啊")]
    rows = []
    for ref, hyp in checks:
        mine_w = error_rate(align(ref.split(), hyp.split()))
        mine_c = error_rate(align(list(ref), list(hyp)))
        jw, jc = jiwer.wer(ref, hyp), jiwer.cer(ref, hyp)
        ok = abs(mine_w - jw) < 1e-9 and abs(mine_c - jc) < 1e-9
        rows.append((ref[:24], hyp[:24], f"{mine_w:.3f}", f"{jw:.3f}", f"{mine_c:.3f}", f"{jc:.3f}", "一致" if ok else "不一致"))
    show_table(["REF", "HYP", "我的 WER", "jiwer WER", "我的 CER", "jiwer CER", "結果"], rows)
    STATUS["jiwer 交叉驗證"] = "real"
except ImportError:
    print("（選用）未安裝 jiwer：!pip -q install jiwer 後重跑本格。")
    STATUS["jiwer 交叉驗證"] = "skipped"

**檢核**
- [ ] 我能手算「今天學習語音 → 今天學習語言」的 CER = 1/6 ≈ 16.7%。
- [ ] 我能解釋為什麼中文常用 CER，以及為什麼計分前要宣告正規化規則。
- [ ] 我自己的實作和 jiwer 的總錯誤率一致。

## 12.14 控制實驗：比較兩個 Whisper 模型

**實驗設計**

| 項目 | 本實驗 |
|---|---|
| 自變項（只改這個）| Whisper 模型：`tiny`、`small` |
| 控制變項 | 同一個音檔 `lecture.wav`、同一份 `reference.txt`、`language="zh"`、同一套正規化規則（`SCORE_RULE`）、同一台裝置 |
| 依變項 | 轉錄時間、CER、專有名詞錯誤 |

朗讀稿：

> 今天測試 Whisper 語音辨識。我們使用 Python、NumPy 與 librosa 分析聲音，STFT 的 Window Size 設定為 2048。

專有名詞清單：Whisper、Python、NumPy、librosa、STFT、Window Size、2048。

> **先預測再執行**：`small` 一定比 `tiny` 準嗎？會慢幾倍？

In [ ]:
TERMS = ["Whisper", "Python", "NumPy", "librosa", "STFT", "Window Size", "2048"]
MODELS = ["tiny", "small"]          # TODO（選做）：加入 "base" 或 "medium"
DEMO_HYP = "今天測試 whisper 語音識別。我們使用 python、numpy 與 Libra 分析聲音，STF 的 window size 設定為 2048。"


def term_misses(hyp, terms, rules=None):
    """回傳辨識結果中「沒有正確出現」的專有名詞（用同一套正規化規則比對）。"""
    rules = RULESETS[SCORE_RULE] if rules is None else rules
    h = normalize(hyp, rules)
    return [t for t in terms if normalize(t, rules) not in h]


def score_row(name, reference, hyp, terms, seconds=None, audio_sec=None):
    c3 = cer(reference, hyp, RULESETS["R3 NFKC＋小寫＋去標點＋去空白"])
    c4 = cer(reference, hyp, RULESETS[SCORE_RULE])
    rtf = f"{seconds / audio_sec:.2f}" if seconds and audio_sec else "—"
    miss = term_misses(hyp, terms)
    return (name, DEVICE, round(seconds, 2) if seconds else "—", rtf, f"{c3:.1%}", f"{c4:.1%}",
            f"{len(miss)}：" + ("、".join(miss) if miss else "無"), hyp.strip())


HEADER_14 = ["模型", "裝置", "轉錄時間 (s)", "即時倍率 RTF", "CER（R3）", "CER（R4）", "專有名詞錯誤", "辨識結果"]


def compare_models(audio, reference, models, terms, prompt=None):
    rows, outs = [], {}
    audio_sec = duration(audio)
    for name in models:
        kwargs = {"initial_prompt": prompt} if prompt else {}
        res, sec = transcribe(audio, name, **kwargs)
        if res is None:
            rows.append((name, DEVICE, "—", "—", "—", "—", "—", "（未執行）"))
            continue
        outs[name] = res
        rows.append(score_row(name + (" + prompt" if prompt else ""), reference, res["text"], terms, sec, audio_sec))
    show_table(HEADER_14, rows)
    return outs, rows


print(f"音檔：lecture.wav（{LECTURE_ENGINE}）｜計分規則：{SCORE_RULE}｜裝置：{DEVICE} {GPU_NAME}")
exp_out, exp_rows = compare_models("lecture.wav", REFERENCE, MODELS, TERMS)
exp_out_p, exp_rows_p = compare_models("lecture.wav", REFERENCE, MODELS, TERMS, prompt=WHISPER_PROMPT)

if not exp_out:
    print("Whisper 未執行。以下用示範假設文字 DEMO_HYP（不是 Whisper 的輸出）展示計分流程：")
    show_table(HEADER_14, [score_row("示範 DEMO_HYP", REFERENCE, DEMO_HYP, TERMS)])
    best_hyp = DEMO_HYP
else:
    best_hyp = exp_out[MODELS[-1]]["text"] if MODELS[-1] in exp_out else next(iter(exp_out.values()))["text"]
show_alignment(align(list(normalize(REFERENCE, RULESETS[SCORE_RULE])), list(normalize(best_hyp, RULESETS[SCORE_RULE]))))

**觀察與檢核**
- [ ] 表格中每一列只差在模型（與是否加 prompt），其他條件相同。
- [ ] 我能用數字說明「大模型一定比較好嗎？」：準確度提升多少、時間多花幾倍、對專有名詞有沒有幫助。
- [ ] 錄音是 TTS 合成時，結果**只代表這個 TTS 聲音**，不代表真人錄音；練習 2 要用你自己的錄音再做一次。

## 12.15 TTS → STT 往返測試

**概念**：把文字丟進 TTS 得到聲音，再丟進 Whisper 變回文字，和原文比 CER。這可以**自動**找出 TTS 念不清楚、或 STT 認不出來的地方——數字、英文、專有名詞通常最先出問題。

但要記住：**辨識得出來 ≠ 聽起來自然**。一個很機械的聲音也可能被辨識得很準；往返測試測不到自然度、情緒、韻律，也分不出錯誤是 TTS 念錯還是 STT 聽錯。

> **先預測再執行**：五種句子中，哪一種的 CER 會最高？

In [ ]:
ROUNDTRIP_ITEMS = [
    ("純中文", "今天天氣很好，我們在教室裡練習聲音剪輯。"),
    ("中英混合", "請打開 Colab，然後執行 librosa 的 STFT。"),
    ("數字", "這段音訊長 3.5 秒，取樣率是 44100 Hz。"),
    ("專有名詞", "Whisper、NumPy 與 OpenCV 都是常見的工具。"),
    ("日期與金額", "2026 年 11 月 9 日，門票 350 元。"),
]
RT_VOICE = "zh-TW-HsiaoChenNeural"
RT_MODEL = "small"


async def round_trip(items, voice=RT_VOICE, model_name=RT_MODEL, tag="rt"):
    rules = RULESETS[SCORE_RULE]
    rows, worst = [], None
    for i, (kind, text) in enumerate(items):
        wav = f"{tag}_{i}.wav"
        eng = await tts_to_wav(text, wav, voice=voice, label=f"{tag}_{i}")
        res, sec = transcribe(wav, model_name)
        if res is None:
            rows.append((kind, text, eng, "（Whisper 未執行）", "—"))
            continue
        hyp = res["text"].strip()
        r = align(list(normalize(text, rules)), list(normalize(hyp, rules)))
        rows.append((kind, text, eng, hyp, f"{error_rate(r):.1%}"))
        if worst is None or error_rate(r) > error_rate(worst[1]):
            worst = (kind, r)
    show_table(["類型", "原文", "TTS 引擎", f"Whisper {model_name} 辨識", "CER"], rows)
    if worst:
        print(f"失真最嚴重：{worst[0]}")
        show_alignment(worst[1])
    if any(r[2] == "placeholder" for r in rows):
        print("注意：有些句子是佔位聲音，CER 沒有意義。")
    return rows


rt_rows = await round_trip(ROUNDTRIP_ITEMS)

**觀察與檢核**
- [ ] 數字常見的錯誤是「格式」不同（3.5 vs 三點五、44100 vs 四萬四千一百）：這算 TTS 錯、STT 錯，還是正規化規則沒定好？
- [ ] 我能找出失真最嚴重的一句，並判斷是 TTS 念錯還是 STT 聽錯（**回去聽那一句**）。
- [ ] 我能說出往返測試測不到的東西：自然度、情緒、韻律、是不是像某個人。

## Z｜ZONOS3-Mini：開放權重的多語 TTS 與語者條件

**ZONOS3-Mini** 是 Zyphra（美國舊金山）於 **2026-11-09** 發布的語音合成模型：

| 項目 | 內容 |
|---|---|
| 授權 | 權重 **Apache 2.0**；執行程式碼 **MIT** |
| 架構 | 稀疏 MoE（mixture of experts），總參數約 **800 M**、每步實際啟用約 **100 M** |
| 語言 | **43 種**，包含華語（語言代碼 **`cmn`**）|
| 輸出 | **44.1 kHz** 波形；也回傳 codec 代碼 `codes`（它不輸出 Mel，而是預測 codec 的離散代碼）|
| 語者 | 27 個內建聲音（release voices），以及 **zero-shot voice cloning**（給一段參考聲音即可模仿）|

本週使用 Hugging Face 上的 **Transformers 封裝版** [`Zyphra/ZONOS3-Mini-hf`](https://huggingface.co/Zyphra/ZONOS3-Mini-hf)，只用模型卡上記載的參數：

| `generate()` 參數 | 說明（預設值）|
|---|---|
| 聲音來源（四選一）| `voice`（內建聲音）｜`speaker_audio` + `speaker_sample_rate`（參考音檔）｜`speaker_embedding`（192 維嵌入）｜`speaker_reference_codes`（最多使用 60 秒）|
| `speaker_mode` | `"reference"` 或 `"embedding"`（內建聲音／codes 預設 reference，音檔／嵌入預設 embedding）|
| `speaker_cfg_scale` | 1.0；大於 1 會加強音色（多跑一次 LM，較慢）|
| `language` | 預設 `"en_us"`；華語用 `"cmn"` |
| `text_normalization` | `True`；安裝 `zyphra-textnorm` 才會把數字、日期、金額、單位轉成念法 |
| 取樣 | `temperature` 0.75、`topk` 80、`top_p` 0.69、`min_p` 0.0 |
| 其他 | `repetition_penalty` 2.4、`seed`、`headroom_db` 3.0、`fade_out_ms` 0.0 |
| 回傳 | `waveform`、`sampling_rate`（44100）、`seed`、`codes` |

**27 個內建聲音**：Cillian、Imogen、Jett、Morgana、Rupert、Sakura、Savannah、Thaddeus、Willow、Wyatt、Gaspard、Liesel、Wolfgang、Ananya、Arjun、Ginevra、Lorenzo、Minjun、Seoyeon、Haoran、Meilin、Leonor、Thiago、Rodrigo、Ximena、Chaitanya、Sravya。每個聲音是**針對某個語言設計的**（華語聲音是 Haoran、Meilin）；拿非華語聲音念 `cmn`，可能帶有口音——這正是要記錄的觀察。

> **這個封裝版沒有的功能**：情緒、語速、逐字（word-level）等**操控（steering）功能不在 HF 封裝中**。想體驗請用 Zyphra 的瀏覽器版 Voice Studio，或 GitHub 上的原生執行環境 <https://github.com/Zyphra/ZONOS3-Mini>。（模型卡另列有 `condition_values`／`condition_buckets` 等低階條件參數，本課不使用。）

**執行環境**：建議 GPU——執行階段 → 變更執行階段類型 → **T4 GPU**。CPU 也能跑，但每句可能要數十秒以上。下載權重需要網路（Hugging Face）。

### Z.1 安裝與載入

`trust_remote_code=True` 代表**會下載並執行模型倉庫裡的 Python 程式碼**。這是使用自訂模型的常見做法，但也是資安風險：只對信任的來源使用，正式專案應固定版本（revision）。

In [ ]:
!pip -q install "transformers>=4.57" torch soundfile
!pip -q install zyphra-textnorm   # 選用但建議：把數字、日期、金額、單位轉成念法

In [ ]:
ZONOS_ID = "Zyphra/ZONOS3-Mini-hf"
zonos = None


def wave_to_np(w):
    """torch tensor 或 numpy → 1-D float32 numpy。"""
    if hasattr(w, "detach"):                 # torch tensor（可能在 GPU 上）
        w = w.detach().to("cpu").float().numpy()
    elif hasattr(w, "numpy"):
        w = w.numpy()
    w = np.asarray(w, dtype=np.float32).squeeze()
    if w.ndim == 2:   # 多聲道 → 取平均
        w = w.mean(axis=0 if w.shape[0] < w.shape[1] else 1)
    return w


try:
    import soundfile as sf
    import torch
    from transformers import AutoModel

    t0 = time.perf_counter()
    model = AutoModel.from_pretrained("Zyphra/ZONOS3-Mini-hf", trust_remote_code=True)
    model = model.to("cuda" if torch.cuda.is_available() else "cpu")
    print(f"載入 {ZONOS_ID}：{time.perf_counter() - t0:.1f} s｜裝置：{DEVICE}｜transformers {pkg_version('transformers')}")

    t0 = time.perf_counter()
    out = model.generate("Hello! This is ZONOS3-Mini running in transformers.", voice="Savannah", seed=0)
    try:
        hello = out.waveform.numpy()
    except Exception:              # 若 waveform 留在 GPU 上，先搬回 CPU
        hello = wave_to_np(out.waveform)
    sf.write("hello.wav", hello, out.sampling_rate)
    print(f"hello.wav：{time.perf_counter() - t0:.1f} s 生成，取樣率 {out.sampling_rate} Hz，seed = {out.seed}")
    zonos = model
    STATUS["ZONOS3-Mini 載入"] = "real"
    play("hello.wav", label="ZONOS3-Mini，voice=Savannah，seed=0")
    print("generate() 的參數簽名：", inspect.signature(zonos.generate))
except Exception as err:
    STATUS["ZONOS3-Mini 載入"] = "fallback"
    print(f"ZONOS3-Mini 無法載入（{type(err).__name__}: {str(err)[:300]}）")
    print("→ 可能原因：沒有網路／無法連到 Hugging Face、transformers 版本太舊、記憶體不足。"
          "\n→ Z 段後面的格子會略過；請改用 GPU 執行階段（T4）並確認網路後重跑。")

In [ ]:
def zonos_tts(text, wav_path, label=None, **gen_kwargs):
    """呼叫 zonos.generate()（只傳模型卡記載的參數），存成 WAV 並記錄到 GEN_LOG。"""
    label = label or Path(wav_path).stem
    if zonos is None:
        STATUS[f"ZONOS {label}"] = "skipped"
        return None
    try:
        t0 = time.perf_counter()
        out = zonos.generate(text, **gen_kwargs)
        sec = time.perf_counter() - t0
        y, sr = wave_to_np(out.waveform), int(out.sampling_rate)
        sf.write(str(wav_path), y, sr, subtype="PCM_16")
        used_seed = int(out.seed) if out.seed is not None else None
        params = ", ".join(f"{k}={v}" for k, v in gen_kwargs.items()
                           if k not in ("speaker_audio", "speaker_embedding", "voice", "seed"))
        GEN_LOG[label] = dict(engine="ZONOS3-Mini", tool=f"{ZONOS_ID}（transformers {pkg_version('transformers')}）",
                              voice=gen_kwargs.get("voice", "（參考聲音／嵌入）"), params=params, seed=used_seed,
                              seconds=round(sec, 2), path=str(wav_path), sr=sr, text=text, date=TODAY)
        STATUS[f"ZONOS {label}"] = "real"
        return y, sr
    except Exception as err:
        print(f"ZONOS 生成失敗（{label}，{type(err).__name__}: {str(err)[:200]}）")
        STATUS[f"ZONOS {label}"] = "fallback"
        return None


print("ZONOS 可用" if zonos is not None else "ZONOS 不可用：以下 Z 段只會印出略過訊息。")

### Z.2 活動 (a)：同一句華語，換不同內建聲音

**概念**：固定文字與 `seed`，只換 `voice`，全部用 `language="cmn"`。華語聲音 Meilin、Haoran 之外，也刻意放入英語聲音 Savannah、Sakura——它們念華語可能帶口音，這是要**記錄下來的觀察**，不是錯誤。

這些版本會和 12.5 的 edge-tts 版本**一起放進同一張盲測表**（固定文字 `BLIND_TEXT`，變因 = 聲音／引擎）。

> **先預測再執行**：在盲測中，你猜得出哪些是 edge-tts、哪些是 ZONOS 嗎？

In [ ]:
Z_VOICES = ["Meilin", "Haoran", "Savannah", "Sakura"]   # TODO（選做）：換成其他內建聲音
Z_SEED = 0
for v in Z_VOICES:
    zonos_tts(BLIND_TEXT, f"blind_zonos_{v}.wav", voice=v, language="cmn", seed=Z_SEED)

z_items = []
for v in ["zh-TW-HsiaoChenNeural", "zh-TW-YunJheNeural"]:      # edge-tts 對照組（同一句、rate=+0%）
    path = "blind_edge_" + v.split("-")[2].replace("Neural", "") + ".wav"
    await tts_to_wav(BLIND_TEXT, path, voice=v)
    z_items.append((f"edge-tts {v}, rate=+0%", path))
z_items += [(f"ZONOS3-Mini voice={v}, language=cmn, seed={Z_SEED}", f"blind_zonos_{v}.wav")
            for v in Z_VOICES if Path(f"blind_zonos_{v}.wav").exists()]
print(f"固定文字：{BLIND_TEXT}\n變因：聲音／引擎（共 {len(z_items)} 個版本）\n")
make_blind_test("mixed", z_items, seed=BLIND_SEED + 1)

**評分表（混合盲測）**：多一欄「口音／語言觀察」。

| 聽者 | 版本 | 自然度 1–5 | 清晰度 1–5 | 韻律 1–5 | 口音／語言觀察 |
|---|---|---|---|---|---|
| 聽者 1 | A | | | | |
| 聽者 1 | B | | | | |
| … | … | | | | |
| 聽者 2 | A | | | | |

In [ ]:
# TODO：(自然度, 清晰度, 韻律)；版本數依上面實際產生的數量
SCORES_MIXED = {
    "listener_1": {k: None for k in BLIND["mixed"]},
    "listener_2": {k: None for k in BLIND["mixed"]},
}
reveal("mixed", SCORES_MIXED)

### Z.3 活動 (b)：`seed` 與 `temperature`

**概念**：ZONOS3-Mini 是**取樣式**生成：每一步從機率分佈中抽一個 codec 代碼。

- `seed` 固定亂數：同樣的 seed → 同樣的結果（可重現）；換 seed → 同一個聲音念出不同的「一次演出（take）」。
- `temperature` 控制分佈的「平坦程度」：低（0.5）→ 比較保守、穩定、可能單調；高（1.0）→ 變化多，但也較容易念錯或出現雜音。

> **先預測再執行**：三個不同 seed 的長度會一樣嗎？`temperature=1.0` 會比 0.5 長還是短？

In [ ]:
rows = []
for s in [0, 1, 2]:
    if zonos_tts(BLIND_TEXT, f"meilin_seed{s}.wav", voice="Meilin", language="cmn", seed=s):
        rows.append((f"seed={s}", "0.75（預設）", round(duration(f"meilin_seed{s}.wav"), 2), GEN_LOG[f"meilin_seed{s}"]["seconds"]))
        play(f"meilin_seed{s}.wav", label=f"Meilin，seed={s}")
for temp in [0.5, 1.0]:
    name = f"meilin_temp{str(temp).replace('.', '')}"
    if zonos_tts(BLIND_TEXT, f"{name}.wav", voice="Meilin", language="cmn", seed=0, temperature=temp):
        rows.append(("seed=0", temp, round(duration(f"{name}.wav"), 2), GEN_LOG[name]["seconds"]))
        play(f"{name}.wav", label=f"Meilin，seed=0，temperature={temp}")
if rows:
    show_table(["seed", "temperature", "長度 (s)", "生成時間 (s)"], rows)
    paths = [f"meilin_seed{s}.wav" for s in [0, 1, 2]] + ["meilin_temp05.wav", "meilin_temp10.wav"]
    paths = [p for p in paths if Path(p).exists()]
    fig, axes = plt.subplots(1, len(paths), figsize=(3 * len(paths), 3), sharey=True)
    for ax, p in zip(np.atleast_1d(axes), paths):
        yy, ss = load_audio(p)
        librosa.display.specshow(librosa.power_to_db(librosa.feature.melspectrogram(y=yy, sr=ss, n_mels=80), ref=np.max),
                                 sr=ss, x_axis="time", y_axis="mel", ax=ax)
        ax.set_title(Path(p).stem, fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("ZONOS 不可用，略過 Z.3。")

**聆聽檢核**
- [ ] 不同 seed：音色相同，但停頓、語調、長度不同。
- [ ] 同一個 seed 重跑一次，應該得到相同的結果（可以自己驗證）。
- [ ] `temperature` 高的版本是否比較「活」？是否出現念錯、重複或雜音？

### Z.4 活動 (c)：語者嵌入與相似度矩陣（12.3）

**概念**：`zonos.embed_speaker(音訊, 取樣率)` 把一段聲音變成 **192 維**的語者嵌入。兩個嵌入的**餘弦相似度** cos θ = a·b / (|a||b|)：越接近 1，模型認為音色越像。

我們把 Z.2、Z.3 與 edge-tts 的輸出都算出嵌入，畫成矩陣。ZONOS 不可用時，改用「MFCC 平均＋標準差」當作很粗糙的替代向量（**只為了讓流程能跑**，鑑別力遠不如訓練過的語者編碼器）。

> **先預測再執行**：Meilin 不同 seed 之間的相似度，會比 Meilin vs Haoran 高多少？

In [ ]:
def speaker_vector(path):
    """有 ZONOS → 192 維語者嵌入；沒有 → MFCC 統計向量（粗糙替代）。"""
    y, sr = load_audio(path)
    if zonos is not None:
        emb = zonos.embed_speaker(y[None, :], sr)
        return wave_to_np(emb).reshape(-1), "ZONOS embed_speaker (192-d)"
    mf = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    return np.concatenate([mf.mean(axis=1), mf.std(axis=1)]), "MFCC mean+std fallback (40-d)"


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def similarity_matrix(paths, title=""):
    vecs, kind = [], ""
    for p in paths:
        v, kind = speaker_vector(p)
        vecs.append(v)
    n = len(paths)
    S = np.array([[cosine(vecs[i], vecs[j]) for j in range(n)] for i in range(n)])
    labels = [Path(p).stem for p in paths]
    plt.figure(figsize=(1.0 * n + 3, 0.8 * n + 2))
    plt.imshow(S, vmin=min(0, S.min()), vmax=1, cmap="viridis")
    plt.colorbar(label="Cosine similarity")
    plt.xticks(range(n), labels, rotation=45, ha="right", fontsize=8)
    plt.yticks(range(n), labels, fontsize=8)
    for i in range(n):
        for j in range(n):
            plt.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=7,
                     color="white" if S[i, j] < 0.7 else "black")
    plt.title(f"{title}\n{kind}", fontsize=10)
    plt.grid(False)
    plt.tight_layout()
    plt.show()
    STATUS["Z.4 相似度矩陣"] = "real" if zonos is not None else "fallback"
    return S, labels


sim_paths = [p for p in ["meilin_seed0.wav", "meilin_seed1.wav", "meilin_seed2.wav",
                         "blind_zonos_Haoran.wav", "blind_zonos_Savannah.wav", "blind_zonos_Sakura.wav",
                         "voice_HsiaoChen.wav", "voice_YunJhe.wav", "voice_HsiaoYu.wav"] if Path(p).exists()]
S_sim, sim_labels = similarity_matrix(sim_paths, "Speaker similarity between synthetic voices")

**觀察**：同一個聲音、不同 seed 的相似度通常最高；不同聲音之間較低。但「多高才算同一人」沒有絕對標準——門檻是應用自己訂的，而且會同時犯「誤認」與「漏認」兩種錯。

### Z.5 活動 (d)：只用**自己的聲音**做 voice cloning

**規則**：

1. 參考聲音必須是**你本人**的錄音（0.3 的 `my_voice.wav`；建議 10–30 秒、安靜環境、單人說話）。
2. 把 `I_CONSENT_THIS_IS_MY_OWN_VOICE` 改成 `True`，代表你確認：錄音中的人就是你，而且你同意用於本課練習。**預設 `False` 時程式會拒絕執行。**
3. 生成的檔案只用於課堂作業，必須附上 Audio Disclosure（Z.6），不得冒充、不得當作真實錄音發布。

講義的兩種寫法都會示範：直接給參考音檔（`speaker_audio`），或先算出嵌入再重複使用（`speaker_embedding`）。

In [ ]:
I_CONSENT_THIS_IS_MY_OWN_VOICE = False   # 只有「錄音中的人就是你本人」時才改成 True

print(SAFETY_BOUNDARY, "\n")
CLONE_TEXT = "今天測試 Whisper 語音辨識。我們使用 Python、NumPy 與 librosa 分析聲音。"
clone_ok = False
if not I_CONSENT_THIS_IS_MY_OWN_VOICE:
    print("拒絕執行 voice cloning：I_CONSENT_THIS_IS_MY_OWN_VOICE = False。"
          "\n只有在 my_voice.wav 是你本人的聲音、而且你同意用於本課練習時，才改成 True。")
    STATUS["Z.5 本人聲音複製"] = "skipped"
elif not Path("my_voice.wav").exists():
    print("找不到 my_voice.wav：請先在 0.3 錄音或上傳你自己的聲音。")
    STATUS["Z.5 本人聲音複製"] = "skipped"
elif zonos is None:
    print("ZONOS3-Mini 未載入，無法執行。")
    STATUS["Z.5 本人聲音複製"] = "skipped"
else:
    wav, sr = sf.read("my_voice.wav", dtype="float32", always_2d=True)
    print(f"參考聲音 my_voice.wav：{len(wav) / sr:.1f} 秒，{sr} Hz")
    ok1 = zonos_tts(CLONE_TEXT, "clone_from_clip.wav", label="clone_from_clip",
                    speaker_audio=wav.T, speaker_sample_rate=sr, language="cmn", seed=1)
    try:
        emb = zonos.embed_speaker(wav.T, sr)   # 192 維，可重複使用
        ok2 = zonos_tts(CLONE_TEXT, "clone_from_embedding.wav", label="clone_from_embedding",
                        speaker_embedding=emb, language="cmn", seed=1)
    except Exception as err:
        print(f"embed_speaker 失敗（{type(err).__name__}: {err}）")
        ok2 = None
    for label in ["clone_from_clip", "clone_from_embedding"]:
        if label in GEN_LOG:
            GEN_LOG[label]["voice"] = "本人聲音（my_voice.wav）"
            GEN_LOG[label]["cloning"] = "使用本人聲音，已同意。"
    clone_ok = bool(ok1 or ok2)
    STATUS["Z.5 本人聲音複製"] = "real" if clone_ok else "fallback"
    play("my_voice.wav", label="本人真實錄音")
    for p in ["clone_from_clip.wav", "clone_from_embedding.wav"]:
        if Path(p).exists():
            play(p, label=f"{p}（合成，非真實錄音）")

### 本人 vs 複製：相似度、往返辨識與「像不像 ≠ 是不是」（12.17）

**概念**：複製的目標就是讓語者嵌入**接近**本人，所以「相似度很高」正是它被設計出來的結果——這也說明**「聽起來像」不能當作身分驗證**：聲紋比對、電話裡「聽聲音認人」，都可能被合成語音騙過。

> **先預測再執行**：本人 vs 複製的相似度，會比本人 vs Meilin 高很多嗎？複製聲音的 Whisper CER 會比本人錄音低還是高？

In [ ]:
if not clone_ok:
    print("沒有複製聲音（未同意、沒有本人錄音或 ZONOS 不可用），略過本格。")
else:
    real_vec, kind = speaker_vector("my_voice.wav")
    rows = []
    if USE_MY_VOICE_FOR_LECTURE and exp_out:      # 本人錄音朗讀 12.14 稿子時，也列出本人的 CER 當基準
        base_res = next(iter(exp_out.values()))
        rows.append(("my_voice（本人，12.14）", "1.000",
                     f"{cer(REFERENCE, base_res['text'], RULESETS[SCORE_RULE]):.1%}", base_res["text"].strip()))
    for p in ["clone_from_clip.wav", "clone_from_embedding.wav", "meilin_seed0.wav", "voice_HsiaoChen.wav"]:
        if not Path(p).exists():
            continue
        sim = cosine(real_vec, speaker_vector(p)[0])
        res, sec = transcribe(p, RT_MODEL)
        ref_text = GEN_LOG.get(Path(p).stem, {}).get("text", "")
        c = f"{cer(ref_text, res['text'], RULESETS[SCORE_RULE]):.1%}" if res and ref_text else "—"
        rows.append((Path(p).stem, f"{sim:.3f}", c, res["text"].strip() if res else "（Whisper 未執行）"))
    show_table(["與本人錄音比較", f"餘弦相似度（{kind}）", f"Whisper {RT_MODEL} CER", "辨識結果"], rows)
    print("提醒：相似度高只代表『模型覺得音色像』，不代表『就是本人』。")

### Z.6 活動 (e)：ZONOS 輸出的 Audio Disclosure

每一段 ZONOS 生成的聲音，都要寫清楚：模型 ID、日期、聲音、seed 與參數；用了本人聲音複製的，要寫「**使用本人聲音，已同意。**」。產生方式在 12.23 的 `audio_disclosure()`，這裡先把需要的資訊列出來：

In [ ]:
z_logs = {k: v for k, v in GEN_LOG.items() if v["engine"] == "ZONOS3-Mini"}
if not z_logs:
    print("本次沒有成功生成的 ZONOS 聲音。")
else:
    show_table(["檔案", "voice", "seed", "參數", "生成時間 (s)", "聲音複製"],
               [(k, v["voice"], v.get("seed"), v["params"], v["seconds"], v.get("cloning", "未使用。"))
                for k, v in z_logs.items()])

## 12.16–12.19 深偽語音：風險、線索與限制

**12.16 為什麼要在意**：Z.5 只用十幾秒的參考聲音就能 zero-shot 複製音色。常見的濫用包括：假冒親友或主管打電話要求轉帳、偽造「錄音證據」、製造名人或政治人物的假發言、騷擾與名譽傷害。

**12.17 「像」不等於「是」**：語者嵌入的相似度只回答「模型覺得音色像不像」。複製技術的目標正是把相似度拉高，所以**聲音像某人，不能證明就是某人**；反過來，同一個人感冒、換麥克風、透過電話，相似度也可能下降。

**12.18 偵測器不是測謊機**：深偽偵測器輸出的是「像不像它看過的合成語音」的**分數**，不是「內容是真是假」：

- 遇到新的生成模型、重新壓縮、電話音質、背景噪音，準確度可能大幅下降（資料分佈改變）。
- 會誤判：某些真人聲音、少見口音或特殊錄音設備可能被判成合成。
- 就算偵測出「是真人錄音」，內容仍可能是斷章取義；合成旁白若有揭露，也不是欺騙。

**12.19 來源證明（provenance）與 C2PA**：C2PA／Content Credentials 是附在檔案上的**數位簽章紀錄**，記載誰、用什麼工具、做過哪些編輯。限制：

- 大多數錄音**沒有**憑證，平台轉傳時也常把它剝除——**沒有憑證 ≠ 是假的**。
- 有憑證只證明「簽署者聲稱了這些事、檔案之後沒被改過」，**不保證內容是真的**；把聲音播放出來再重新錄一次（所謂「類比漏洞」）就能繞過。

### 可觀察線索：真實錄音 vs 合成語音

下面計算幾個**可量測的線索**（不是證據）：

| 線索 | 意思 | 小心 |
|---|---|---|
| 有效頻寬（平均頻譜高於峰值 −60 dB 的最高頻率）| edge-tts 是 24 kHz 取樣的 MP3 → 最多到 12 kHz；真實麥克風錄音的底噪通常會填滿到更高頻 | 電話、壓縮音檔一樣是窄頻 |
| 停頓比例 | 靜音佔整段的比例（W11 的 split／detect_nonsilent）| 朗讀稿 vs 自然說話本來就不同 |
| 基頻變化（半音標準差）| 語調起伏 | 有人天生說話平板 |
| 底噪（最安靜 5% 影格的 RMS）| 真實錄音通常有房間噪音；合成常常幾乎是數位靜音 | 降噪後的真實錄音也很乾淨 |

> **先預測再執行**：TTS 樣本的底噪會比你的真實錄音低多少 dB？

In [ ]:
def audio_clues(path):
    """回傳可觀察線索（只是線索，不是證據）。"""
    y, sr = load_audio(path)
    power = (np.abs(librosa.stft(y, n_fft=2048, hop_length=512)) ** 2).mean(axis=1)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
    db = 10 * np.log10(power + 1e-20)
    above = np.nonzero(db > db.max() - 60)[0]          # 平均頻譜高於峰值 -60 dB 的頻率
    bw = float(freqs[above[-1]]) if len(above) else 0.0
    spans = librosa.effects.split(y, top_db=35)
    voiced = sum(int(e - s) for s, e in spans) / max(len(y), 1)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    f0 = librosa.yin(y, fmin=65, fmax=400, sr=sr, frame_length=2048, hop_length=512)
    n = min(len(f0), len(rms))
    mask = rms[:n] > np.percentile(rms[:n], 60)
    f0v = f0[:n][mask]
    semis = 12 * np.log2(f0v / np.median(f0v)) if len(f0v) else np.array([np.nan])
    return {"檔案": Path(path).name, "取樣率 (Hz)": sr, "長度 (s)": round(len(y) / sr, 2),
            "有效頻寬 (Hz)": int(bw), "停頓比例": f"{1 - voiced:.0%}", "語音段數": len(spans),
            "基頻中位數 (Hz)": round(float(np.median(f0v)), 1) if len(f0v) else "—",
            "基頻變化 (半音 SD)": round(float(np.std(semis)), 2),
            "底噪 (dBFS)": round(float(20 * np.log10(np.percentile(rms, 5) + 1e-10)), 1)}


def compare_clues(paths, title="Mel spectrograms"):
    paths = [p for p in paths if Path(p).exists()]
    clues = [audio_clues(p) for p in paths]
    show_table(list(clues[0].keys()), [list(c.values()) for c in clues])
    fig, axes = plt.subplots(1, len(paths), figsize=(4.2 * len(paths), 3.2), squeeze=False)
    for ax, p in zip(axes[0], paths):
        yy, ss = load_audio(p)
        librosa.display.specshow(librosa.power_to_db(librosa.feature.melspectrogram(y=yy, sr=ss, n_mels=96), ref=np.max),
                                 sr=ss, x_axis="time", y_axis="mel", ax=ax)
        ax.set_title(Path(p).name, fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return clues


clue_paths = ["my_voice.wav", "script_tts.wav", "blind_zonos_Meilin.wav", "clone_from_clip.wav"]
if not Path("my_voice.wav").exists():
    print("沒有本人錄音：先只比較合成樣本。錄了 my_voice.wav 之後重跑本格。")
CLUES = compare_clues(clue_paths, "Real recording vs synthetic speech")

## 12.20 證據卡（Deepfake Evidence Card）

**概念**：判斷一段聲音時，不要只寫「真／假」，而是把證據分成三類，再寫出**還不確定的地方**與**信心程度**：

1. **可觀察訊號**：聽感、頻譜、上面的數值線索。
2. **來源與脈絡**：誰先發布、何時、在哪個平台、有沒有被轉傳或剪輯、內容和時空背景是否一致。
3. **外部查證**：向當事人或官方管道確認、找其他獨立來源、內容憑證（C2PA）、偵測工具的結果（只當參考）。

**證據卡範本**（可直接複製到報告）：

```markdown
# 聲音證據卡
- 樣本：（檔名、長度、取得來源）
- 待查證的說法：（例如「這段是某某人本人在某時某地說的」）

## 1. 可觀察訊號
- 聽感：
- 頻譜／波形：
- 數值線索（頻寬、停頓比例、基頻變化、底噪）：

## 2. 來源與脈絡
- 第一個發布者、時間、平台：
- 是否經過轉傳／重新壓縮／剪輯：
- 內容與說話者、時間地點是否一致：

## 3. 外部查證
- 向當事人或官方管道查證：
- 其他獨立來源：
- 內容憑證（C2PA）／偵測工具結果（僅供參考）：

## 4. 仍無法確定的部分
-

## 5. 判斷與信心程度
- 判斷：較可能是合成／較可能是真實錄音／無法判斷
- 信心程度：低／中／高（理由：）
```

下面用一個 Python 字典產生同樣格式的證據卡。示範樣本是我們自己生成的 `script_tts.wav`——現實中你通常**不知道答案**，所以「來源與脈絡」「外部查證」往往比聽感更重要。

In [ ]:
def evidence_card_md(card):
    """dict → 證據卡 Markdown。"""
    def bullets(items):
        return "\n".join(f"- {x}" for x in items) if items else "- （尚未填寫）"
    return "\n".join([
        f"# 聲音證據卡：{card['sample']}",
        f"- **待查證的說法**：{card['claim']}",
        "", "## 1. 可觀察訊號", bullets(card["observable_signals"]),
        "", "## 2. 來源與脈絡", bullets(card["source_context"]),
        "", "## 3. 外部查證", bullets(card["external_verification"]),
        "", "## 4. 仍無法確定的部分", bullets(card["remaining_uncertainty"]),
        "", "## 5. 判斷與信心程度",
        f"- **判斷**：{card['judgement']}",
        f"- **信心程度**：{card['confidence']}（{card['confidence_reason']}）",
    ])


c = audio_clues("script_tts.wav")
demo_card = {
    "sample": "script_tts.wav",
    "claim": "這是一位老師在課堂上朗讀的真實錄音。",
    "observable_signals": [
        f"取樣率 {c['取樣率 (Hz)']} Hz（奈奎斯特頻率 {c['取樣率 (Hz)'] // 2} Hz），有效頻寬約 {c['有效頻寬 (Hz)']} Hz。",
        f"底噪 {c['底噪 (dBFS)']} dBFS，接近數位靜音，聽不到房間的環境聲。",
        f"基頻變化 {c['基頻變化 (半音 SD)']} 半音；停頓位置整齊、每句結尾的語調很相似。",
    ],
    "source_context": [
        "檔案由本 notebook 產生，沒有經過轉傳。",
        "檔名與 GEN_LOG 顯示引擎為 " + GEN_LOG.get("script_tts", {}).get("engine", "?") + "。",
    ],
    "external_verification": [
        "可以比對 GEN_LOG 的生成紀錄與 Audio Disclosure。",
        "沒有 C2PA 憑證——但這本身不能證明真假。",
    ],
    "remaining_uncertainty": [
        "窄頻與低底噪也可能來自電話錄音或降噪處理。",
        "如果只拿到檔案、沒有生成紀錄，光靠聽感無法確定。",
    ],
    "judgement": "較可能是合成",
    "confidence": "高",
    "confidence_reason": "有生成紀錄可對照；若只有聲音本身，信心只能是「中」",
}
display(Markdown(evidence_card_md(demo_card)))

## 12.23 Audio Disclosure：為作品標示合成聲音

**概念**：只要作品中有合成語音，就附上一段**揭露說明**。講義的範本：

```markdown
## Audio Disclosure
### Synthetic Speech
本作品旁白使用 TTS 生成。
### Tool
edge-tts
### Voice
zh-TW-________
### Input
由作者撰寫之旁白稿。
### Post-processing
- Trim
- Gain Adjustment
- Fade
- Background Music Mixing
### Human Voice Cloning
未使用。
### Final Review
字幕與旁白內容已人工確認。
```

使用 API 或模型時，還要加上**模型版本、日期、參數**，別人才能重現或查證。下面的 `audio_disclosure()` 從變數產生這段文字；不給額外資訊時，輸出必須和範本**完全相同**（程式會自我檢查）。

In [ ]:
DISCLOSURE_TEMPLATE = """## Audio Disclosure
### Synthetic Speech
本作品旁白使用 TTS 生成。
### Tool
edge-tts
### Voice
zh-TW-________
### Input
由作者撰寫之旁白稿。
### Post-processing
- Trim
- Gain Adjustment
- Fade
- Background Music Mixing
### Human Voice Cloning
未使用。
### Final Review
字幕與旁白內容已人工確認。"""


def audio_disclosure(tool="edge-tts", voice="zh-TW-________", input_desc="由作者撰寫之旁白稿。",
                     post=("Trim", "Gain Adjustment", "Fade", "Background Music Mixing"),
                     cloning="未使用。", review="字幕與旁白內容已人工確認。",
                     synthetic="本作品旁白使用 TTS 生成。",
                     model_version=None, date_str=None, params=None):
    lines = ["## Audio Disclosure", "### Synthetic Speech", synthetic, "### Tool", tool, "### Voice", voice]
    if model_version:
        lines += ["### Model Version", model_version]
    if params:
        lines += ["### Parameters", params]
    if date_str:
        lines += ["### Date", date_str]
    lines += ["### Input", input_desc, "### Post-processing", *[f"- {p}" for p in post],
              "### Human Voice Cloning", cloning, "### Final Review", review]
    return "\n".join(lines)


assert audio_disclosure() == DISCLOSURE_TEMPLATE, "與講義範本不一致"
print("audio_disclosure() 預設輸出與講義範本完全相同\n")

log = GEN_LOG.get("script_tts", GEN_LOG["speech"])   # 以 12.14 的朗讀稿旁白為例
edge_disc = audio_disclosure(tool="edge-tts（Microsoft Edge 線上 TTS 的第三方用戶端）", voice=log["voice"],
                             model_version=log["tool"], params=log["params"], date_str=log["date"],
                             post=("Trim", "Fade"))
if log["engine"] != "edge-tts":
    edge_disc += "\n\n> 注意：本次執行 edge-tts 未連線，實際檔案是離線佔位聲音，不可作為作品使用。"
Path("disclosure_edge.md").write_text(edge_disc, encoding="utf-8")
display(Markdown(edge_disc))

In [ ]:
# ZONOS3-Mini 的揭露（活動 e）：模型 ID、日期、voice、seed 與參數；本人聲音複製要寫明已同意
z_logs = {k: v for k, v in GEN_LOG.items() if v["engine"] == "ZONOS3-Mini"}
pick = GEN_LOG.get("clone_from_clip") or GEN_LOG.get("blind_zonos_Meilin") or next(iter(z_logs.values()), None)
if pick is None:
    print("本次沒有 ZONOS 生成結果；以下是填好欄位的範例（實際使用時以 GEN_LOG 為準）：\n")
    pick = dict(tool=f"{ZONOS_ID}（transformers {pkg_version('transformers')}）", voice="Meilin", seed=0,
                params="language=cmn, temperature=0.75（預設）", date=TODAY)
zonos_disc = audio_disclosure(
    tool="ZONOS3-Mini（Zyphra，開放權重 Apache 2.0）",
    voice=pick["voice"],
    model_version=pick["tool"],
    params=f"seed={pick.get('seed')}, {pick['params']}",
    date_str=pick["date"],
    post=("Trim", "Gain Adjustment", "Fade"),
    cloning=pick.get("cloning", "未使用。"),
)
Path("disclosure_zonos.md").write_text(zonos_disc, encoding="utf-8")
display(Markdown(zonos_disc))

**檢核**
- [ ] 每一段合成聲音都能在 `GEN_LOG` 找到工具、聲音、參數、日期。
- [ ] 用了本人聲音複製的作品，`Human Voice Cloning` 寫的是「使用本人聲音，已同意。」
- [ ] 我知道 Disclosure 是給聽眾看的，不能藏在檔案深處。

## 練習 1：TTS 控制變因盲測

1. 修改 `TODO`：自訂固定文字、選擇**一個**變因（`voice` 或 `rate`）、設定三個版本與 seed。
2. 找**至少兩位**聽者，在不知道代號的情況下評分，填進 `SCORES_EX1`，再執行揭曉。
3. 寫結論，並附上 Audio Disclosure。

| 項目 | 內容 |
|---|---|
| 固定文字 | |
| 變因（只有一個）| |
| 版本 1／2／3 的設定 | |
| 打亂 seed | |
| 播放裝置與音量 | |
| 聽者人數 | |
| 結論（只談這個變因）| |

In [ ]:
# TODO 1：固定文字
EX1_TEXT = "各位同學好，這是我的語音合成盲測，請依照自然度、清晰度與韻律評分。"
# TODO 2：只改一個變因，三個版本
EX1_CONDITIONS = [dict(voice="zh-TW-HsiaoChenNeural", rate="+0%"),
                  dict(voice="zh-TW-YunJheNeural", rate="+0%"),
                  dict(voice="zh-TW-HsiaoYuNeural", rate="+0%")]
EX1_SEED = 115

ex1_items = []
for i, cond in enumerate(EX1_CONDITIONS):
    await tts_to_wav(EX1_TEXT, f"ex1_{i}.wav", **cond)
    ex1_items.append((f"{cond['voice']}, rate={cond['rate']}", f"ex1_{i}.wav"))
changed = [k for k in EX1_CONDITIONS[0] if len({c[k] for c in EX1_CONDITIONS}) > 1]
print("有變動的參數：", changed, "← 應該只有一個" if len(changed) == 1 else "← 警告：變因不只一個！")
make_blind_test("ex1", ex1_items, seed=EX1_SEED)

In [ ]:
# TODO 3：(自然度, 清晰度, 韻律)，至少兩位聽者
SCORES_EX1 = {
    "listener_1": {"A": None, "B": None, "C": None},
    "listener_2": {"A": None, "B": None, "C": None},
}
reveal("ex1", SCORES_EX1)
lg = GEN_LOG["ex1_0"]
display(Markdown(audio_disclosure(voice="、".join(c["voice"] for c in EX1_CONDITIONS), model_version=lg["tool"],
                                  params="；".join(f"rate={c['rate']}" for c in EX1_CONDITIONS),
                                  date_str=lg["date"], post=("Trim",))))

## 練習 2：Whisper 準確度（自己的 60 秒錄音）

1. 在 0.3 把 `RECORD_SECONDS` 改成 `60` 錄音（或上傳），內容自訂，但要包含**至少 3 個專有名詞**與一些數字。
2. 逐字整理參考稿 `EX2_REFERENCE`（以你實際念出來的內容為準），存成 `ex2_reference.txt`。
3. 至少比較兩個模型，填寫下表。

| 模型 | 裝置 | 轉錄時間 (s) | CER | 專有名詞錯誤 | 主要錯誤類型（S／D／I）|
|---|---|---|---|---|---|
| tiny | | | | | |
| small | | | | | |

正規化規則：`SCORE_RULE` = ＿＿＿＿（若修改，請寫明）

In [ ]:
EX2_AUDIO = "my_voice.wav"                       # TODO：你的 60 秒錄音
EX2_REFERENCE = """TODO：貼上你逐字整理的參考稿"""
EX2_TERMS = ["TODO"]                             # TODO：錄音中的專有名詞
EX2_MODELS = ["tiny", "small"]

if not Path(EX2_AUDIO).exists():
    print(f"找不到 {EX2_AUDIO}：請先在 0.3 錄音或上傳。")
elif "TODO" in EX2_REFERENCE:
    print("請先填寫 EX2_REFERENCE 與 EX2_TERMS。")
else:
    print(f"{EX2_AUDIO}：{duration(EX2_AUDIO):.1f} 秒（建議約 60 秒）")
    Path("ex2_reference.txt").write_text(EX2_REFERENCE.strip() + "\n", encoding="utf-8")
    ex2_out, ex2_rows = compare_models(EX2_AUDIO, EX2_REFERENCE, EX2_MODELS, EX2_TERMS)
    for name, res in ex2_out.items():
        r = align(list(normalize(EX2_REFERENCE, RULESETS[SCORE_RULE])), list(normalize(res["text"], RULESETS[SCORE_RULE])))
        print(f"{name}: S={r['S']} D={r['D']} I={r['I']} N={r['N']}")

## 練習 3：TTS → STT 往返測試

1. 自訂至少 5 句，分別包含：純中文、中英混合、數字、專有名詞、你的領域用語。
2. 執行後找出**失真最嚴重**的一句，回去聽它，判斷是 TTS 念錯、STT 聽錯，還是正規化規則的問題。

| 類型 | 原文 | 辨識結果 | CER | 問題出在哪裡？|
|---|---|---|---|---|
| | | | | |

> 提醒：CER 低只代表「辨識得出來」，不代表自然。請另寫一句你對自然度的聆聽評語。

In [ ]:
EX3_ITEMS = [                                   # TODO：換成你自己的句子
    ("純中文", "期末專題要繳交一支三分鐘的短影片。"),
    ("中英混合", "我們用 Python 的 pydub 做 Podcast 剪輯。"),
    ("數字", "影片解析度是 1920 乘 1080，每秒 30 格。"),
    ("專有名詞", "Stable Diffusion 與 LoRA 是生成影像的工具。"),
    ("領域用語", "請把取樣率設定為 48 kHz，位元深度 24 bit。"),
]
EX3_VOICE = "zh-TW-HsiaoChenNeural"
EX3_MODEL = "small"
ex3_rows = await round_trip(EX3_ITEMS, voice=EX3_VOICE, model_name=EX3_MODEL, tag="ex3")

## 練習 4：證據卡（真實自錄 vs 合成）

比較**你自己的真實錄音** `my_voice.wav` 與一段**合成語音**（edge-tts 的 `script_tts.wav`；若做了 Z.5，也可以用 `clone_from_clip.wav`）。

1. 執行下一格取得數值線索與 Mel 頻譜。
2. 假裝你**不知道**哪個是真的，為合成樣本填寫一張證據卡（三類證據＋仍不確定的部分＋信心程度）。
3. 回答：如果這段聲音是在社群媒體上看到的、沒有任何生成紀錄，你的信心程度會改變嗎？

In [ ]:
EX4_REAL = "my_voice.wav"
EX4_SYNTH = "clone_from_clip.wav" if Path("clone_from_clip.wav").exists() else "script_tts.wav"   # TODO：可改
if Path(EX4_REAL).exists():
    ex4_clues = compare_clues([EX4_REAL, EX4_SYNTH], "Exercise 4: real vs synthetic")
else:
    print("沒有 my_voice.wav：請先在 0.3 錄音；以下只分析合成樣本。")
    ex4_clues = compare_clues([EX4_SYNTH], "Exercise 4: synthetic only")

ex4_card = {                                   # TODO：每一項都改成你自己的觀察
    "sample": EX4_SYNTH,
    "claim": "TODO：這段聲音被宣稱是什麼？",
    "observable_signals": ["TODO：聽感", "TODO：頻譜／波形", "TODO：數值線索（引用上表）"],
    "source_context": ["TODO：來源、時間、平台、是否轉傳"],
    "external_verification": ["TODO：你會怎麼向外部查證？"],
    "remaining_uncertainty": ["TODO：哪些事情仍無法確定？"],
    "judgement": "TODO：較可能是合成／較可能是真實錄音／無法判斷",
    "confidence": "TODO：低／中／高",
    "confidence_reason": "TODO：理由",
}
display(Markdown(evidence_card_md(ex4_card)))
_ = Path("ex4_evidence_card.md").write_text(evidence_card_md(ex4_card), encoding="utf-8")

## 延伸 1：自動字幕產生器（SRT，選用燒錄）

流程：Whisper（`small` + `initial_prompt`）→ segments → `s2twp` 轉繁體 → `subtitles.srt`。想把字幕「燒」進影片，把 `BURN_IN` 改成 `True`：會安裝 Noto CJK 中文字型，再用 ffmpeg 的 `subtitles` 濾鏡做出黑底字幕影片 `subtitled.mp4`。

> 自動字幕一定要**人工校對**（12.9）；而用了 `s2twp` 也要在說明中寫出來（12.10）。

In [ ]:
SUB_AUDIO = "lecture.wav"
SUB_MODEL = "small"
BURN_IN = False   # 改成 True：產生燒錄字幕的影片

sub_res, sub_sec = transcribe(SUB_AUDIO, SUB_MODEL, initial_prompt=WHISPER_PROMPT)
sub_segments = sub_res["segments"] if sub_res else DEMO_SEGMENTS
if not sub_res:
    print("（Whisper 未執行：使用示範段落）")
print(segments_to_srt(sub_segments, "subtitles.srt", convert=s2twp))

if BURN_IN:
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=False)
    cmd = ["ffmpeg", "-y", "-loglevel", "error",
           "-f", "lavfi", "-i", "color=c=black:s=1280x720:r=25", "-i", SUB_AUDIO,
           "-vf", "subtitles=subtitles.srt:force_style='FontName=Noto Sans CJK TC,FontSize=28'",
           "-shortest", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "subtitled.mp4"]
    done = subprocess.run(cmd, capture_output=True, text=True)
    print("已產生 subtitled.mp4" if done.returncode == 0 else f"ffmpeg 失敗：{done.stderr[-300:]}")

## 延伸 2：CER 與正規化規則

把 12.14 每個模型的辨識結果，用 R0–R4 五種規則分別計分。觀察：**模型的排名會不會因為規則而改變？**

In [ ]:
hyps = {name: res["text"] for name, res in {**exp_out}.items()}
hyps.update({f"{name} + prompt": res["text"] for name, res in exp_out_p.items()})
if not hyps:
    hyps = {"示範 DEMO_HYP": DEMO_HYP}
    print("（Whisper 未執行：使用示範文字）")
rows = [(name, *[f"{cer(REFERENCE, h, rules):.1%}" for rules in RULESETS.values()]) for name, h in hyps.items()]
show_table(["辨識結果", *RULESETS.keys()], rows)

## 延伸 3：Speech AI 品質矩陣

把「主觀分數」與「客觀量測」放在同一張表：自然度、清晰度（來自 Z.2 混合盲測 `SCORES_MIXED`）、STT CER（Whisper 辨識 `BLIND_TEXT`）、生成時間與即時倍率（生成時間 ÷ 音訊長度；< 1 代表比即時快）。edge-tts 與 ZONOS3-Mini 都列為一列。

> 想一想：生成時間比較 edge-tts（雲端服務，含網路延遲）與 ZONOS3-Mini（本地 GPU）公平嗎？要怎麼寫才不誤導？

In [ ]:
path_to_letter = {v["path"]: k for k, v in BLIND.get("mixed", {}).items()}
rows = []
for label, g in GEN_LOG.items():
    if g.get("text") != BLIND_TEXT or not label.startswith("blind_") or g["path"] not in path_to_letter:
        continue
    letter = path_to_letter[g["path"]]
    got = [s[letter] for s in SCORES_MIXED.values() if s.get(letter)]
    nat = round(float(np.mean([x[0] for x in got])), 2) if got else "—"
    intel = round(float(np.mean([x[1] for x in got])), 2) if got else "—"
    res, _ = transcribe(g["path"], RT_MODEL)
    c = f"{cer(BLIND_TEXT, res['text'], RULESETS[SCORE_RULE]):.1%}" if res else "—"
    dur = duration(g["path"])
    rtf = f"{g['seconds'] / dur:.2f}" if g.get("seconds") else "—"
    rows.append((g["engine"], g["voice"], letter, nat, intel, c, g.get("seconds", "—"), rtf, g["sr"]))
show_table(["引擎", "聲音", "盲測代號", "自然度", "清晰度", f"STT CER（{RT_MODEL}）", "生成時間 (s)", "即時倍率", "取樣率"], rows)

## 執行摘要

列出本次執行中，每個需要網路或模型的步驟是**真的執行**（real）、**用了替代方案**（fallback）還是**略過**（skipped）。報告中只能引用 real 的結果。

In [ ]:
show_table(["步驟", "狀態"], sorted(STATUS.items()))
n_fallback = sum(v != "real" for v in STATUS.values())
print(f"real {sum(v == 'real' for v in STATUS.values())} 項｜fallback／skipped {n_fallback} 項")

## 檢核題

請用自己的話回答（每題 2–4 句），盡量引用本 notebook 的數字、圖或聆聽結果。

1. TTS 的輸入與輸出各是什麼？中間的 Mel 頻譜和 W10 的 STFT 頻譜圖有什麼相同、什麼不同？
2. vocoder 在 TTS 中負責什麼？為什麼 Griffin-Lim 聽起來有金屬感？為什麼不是所有 TTS 都使用 Mel（以 ZONOS3-Mini 為例）？
3. TTS、語者條件 TTS（voice cloning）與語音轉換（VC），輸入和輸出各有什麼不同？
4. 在 Colab 本地執行 Whisper，相較於把錄音上傳到雲端 API，隱私上的好處是什麼？還有什麼要注意？
5. 較大的 Whisper 模型一定比較好嗎？請用你在 12.14 或練習 2 的時間與 CER 說明。
6. 為什麼 Whisper 的時間戳不能當作標準答案？`initial_prompt` 能做到什麼、做不到什麼？
7. 中文為什麼常用 CER 而不是 WER？為什麼計分前一定要先宣告正規化規則？請舉一個同一對句子在不同規則下 CER 不同的例子。
8. TTS → STT 往返測試的 CER 很低，能不能說這個 TTS 很自然？往返測試測不到哪些東西？
9. 本人與複製聲音的語者嵌入相似度很高，為什麼不能當作身分驗證？深偽偵測器為什麼不是測謊機？
10. 一段音檔沒有 C2PA／Content Credentials，能不能說它是假的？有的話，能不能說內容是真的？
11. 為什麼本課即使對方口頭同意，也禁止複製第三方的聲音？

### 我的答案

1.
2.
3.
4.
5.
6.
7.
8.
9.
10.
11.

## 繳交清單

- [ ] 本 notebook（`.ipynb`），已執行完成並保留輸出；「執行摘要」中報告引用的步驟都是 real
- [ ] 練習 1：盲測設定表（固定文字、唯一變因、三個版本、seed）、**至少兩位聽者**的評分、揭曉結果、結論與 Audio Disclosure；附 `blind_key_ex1.json`
- [ ] 練習 2：約 60 秒本人錄音的參考稿 `ex2_reference.txt`、**至少兩個模型**的轉錄時間、CER（或 WER）、專有名詞錯誤表，並寫明正規化規則
- [ ] 練習 3：TTS → STT 往返表（至少 5 句）、失真最嚴重的一句與原因分析、一句自然度聆聽評語
- [ ] 練習 4：證據卡 `ex4_evidence_card.md`（三類證據＋仍不確定的部分＋信心程度）
- [ ] 所有合成聲音的 Audio Disclosure（`disclosure_edge.md`、`disclosure_zonos.md`；若做了本人聲音複製，寫明「使用本人聲音，已同意。」）
- [ ] （延伸）`subtitles.srt`、CER 與正規化比較表、Speech AI 品質矩陣
- [ ] 檢核題 11 題作答

> **不要繳交**任何第三方的聲音。你自己的錄音檔（`my_voice.wav`）是否附上由你決定；它就是可以複製你聲音的素材，請不要公開分享。下載檔案：在 Colab 左側「檔案」面板對檔案按右鍵 → 下載。